# Auxiliary Data Resolution Check -- Table 3.2 verification

This settles the Dissertation Plan's Table 3.2 ("Key spatial data resolutions over Aberfoyle")
with **empirical extraction against real plot coordinates**, not stated/nominal resolution
alone -- the plan currently rules some datasets in or out based on a documented grid size,
which isn't good enough evidence on its own (a nominal "1km" or "50m" product can still be a
flat polygon conversion with almost no real local variation, or -- as turns out below -- the
opposite: a coarse-looking grid that's actually terrain-conditioned and does vary locally).

**Every dataset below is tested the same way**, on the same plot set and bounding box, so the
comparison is apples-to-apples:

1. Can the data actually be reached (live API/download test, with the real access method and
   any auth/account it needs)?
2. What does it actually look like at every real Aberfoyle plot -- not a spot-check tile, the
   full analysis plot set?
3. Does it vary enough to be *statistically* useful (not just "has more than one value"), via a
   proper screening battery (coefficient of variation, variogram range, Moran's I, within- vs
   between-compartment variance, and correlation against the already-fitted CR residual) --
   not just eyeballing a distinct-value count.
4. Which of the six survey years (2002/2006/2008/2012/2021/2023) does it actually cover?

Plus five brand-new candidate features derived entirely from the existing GPKG/DTM -- no
external source, so (unlike everything above) they trivially cover all six years.


## Setup

Analysis plot set: every distinct plot in the **4survey cohort** (71,766 plots) -- confirmed
below that the 6survey cohort (13,897 plots) is a strict subset of it, so this one set already
covers both cohorts, matching the task's "extract at every distinct plot coordinate (both
cohorts)" instruction without double-counting. The 6survey cohort is used separately, later,
wherever a *second*, independent CR-residual cross-check is worth having (this repo's
established convention -- see `documentation/progress_notes.md` -- is to never trust a single
cohort's result alone).


In [ ]:
import sys
import time
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.transform
import requests
from pyproj import Transformer
from rasterio.windows import from_bounds
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree
from scipy.stats import spearmanr

# Make the models/ package importable -- same project-root-finding pattern as every other
# notebook in this repo.
notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.chapman_richards.chapman_richards import chapman_richards
from models.common.geo import load_compartment_boundaries, load_plot_coordinates

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

BBOX_27700 = dict(minx=235_000, miny=692_500, maxx=255_000, maxy=707_500)
CACHE_DIR = project_root / "data" / "raw" / "environmental"  # gitignored, same cache used by
CACHE_DIR.mkdir(parents=True, exist_ok=True)                  # notebooks/environmental_data/

print("project_root:", project_root)


In [ ]:
coordinates_df = load_plot_coordinates()             # identification, cpmt, x, y (EPSG:27700)
compartment_boundaries = load_compartment_boundaries()  # cpmt, geometry (EPSG:27700)

master_4survey = pd.read_parquet(project_root / "data/processed/master/clean_master_4survey.parquet")
master_6survey = pd.read_parquet(project_root / "data/processed/master/clean_master_6survey.parquet")

ids_4survey = set(master_4survey["identification"].unique())
ids_6survey = set(master_6survey["identification"].unique())
print(f"4survey: {len(ids_4survey):,} plots, 6survey: {len(ids_6survey):,} plots, "
      f"6survey subset of 4survey: {ids_6survey.issubset(ids_4survey)}")

# The shared analysis plot set used for every dataset below.
plots = coordinates_df[coordinates_df["identification"].isin(ids_4survey)].reset_index(drop=True)
print(f"Analysis plot set: {len(plots):,} distinct plots")


## CR residuals (both cohorts) -- the correlation target for the statistical screen

Recomputed directly from each cohort's own frozen Chapman-Richards fit
(`outputs/chapman_richards/<cohort>/params.json`), applied to **every** plot-year row (not just
the test split `predictions.csv` already has) -- gives a mean residual for every plot in the
analysis set, not just the ~20% held out as test. Mean residual per plot (`epsilon_bar`) is
exactly the quantity the plan's own section 4.1 trajectory classification is defined on.


In [ ]:
import json


def compute_mean_cr_residual(master_df, cohort):
    params = json.loads((project_root / f"outputs/chapman_richards/{cohort}/params.json").read_text())
    predicted = chapman_richards(master_df["Age"].values, params["y_max"], params["k"], params["p"])
    residual = master_df["Top_Height99"].values - predicted
    return (
        pd.DataFrame({"identification": master_df["identification"], "residual": residual})
        .groupby("identification")["residual"].mean()
        .rename("mean_cr_residual")
    )


mean_residual_4survey = compute_mean_cr_residual(master_4survey, "4survey")
mean_residual_6survey = compute_mean_cr_residual(master_6survey, "6survey")

plots = plots.merge(mean_residual_4survey, on="identification", how="left")
plots = plots.rename(columns={"mean_cr_residual": "mean_cr_residual_4survey"})
plots = plots.merge(mean_residual_6survey.rename("mean_cr_residual_6survey"), on="identification", how="left")

print(f"4survey residual: n={plots['mean_cr_residual_4survey'].notna().sum():,}, "
      f"mean={plots['mean_cr_residual_4survey'].mean():.2f}m")
print(f"6survey residual: n={plots['mean_cr_residual_6survey'].notna().sum():,} "
      f"(subset -- only plots that are ALSO in the 6survey cohort)")


## Reusable spatial plot: matplotlib hexbin

Reusing the "quick/rough-story tier" winner from
`data_exploration_gpkg/notebooks/spatial_viz_comparison_scratch.ipynb` section 7 -- that
comparison's own conclusion for exactly this job ("does this even show a spatial pattern,"
fast to build, zero new dependencies, one-line static export) was plain matplotlib `hexbin`,
not any of the interactive candidates it also tried. Same `draw_survey_outline()` backdrop
pattern as that notebook and `results_notebooks/baseline_results.ipynb`.


In [ ]:
def compartments_used_by_plots(plot_ids):
    used_cpmts = coordinates_df.loc[coordinates_df["identification"].isin(plot_ids), "cpmt"].unique()
    return compartment_boundaries[compartment_boundaries["cpmt"].isin(used_cpmts)]


survey_outline = compartments_used_by_plots(plots["identification"])
x_lim = (plots["x"].min() - 500, plots["x"].max() + 500)
y_lim = (plots["y"].min() - 500, plots["y"].max() + 500)


def draw_survey_outline(ax):
    # zorder=3 (above the hexbin's zorder=2) so boundaries sit ON TOP of the colour fill,
    # not hidden underneath it -- light grey so it reads as a boundary, not a heavy overlay.
    survey_outline.plot(ax=ax, facecolor="none", edgecolor="#d5d5d5", linewidth=0.6, zorder=3)


def quick_spatial_plot(df, value_col, title, cmap="viridis", gridsize=110):
    # cmap defaults to a sequential scale (magnitude covariates like elevation/wind speed/pH),
    # not the RdBu_r diverging scale used for signed CR residuals elsewhere in this repo.
    values = df[value_col].dropna()
    df_valid = df.loc[values.index]
    fig, ax = plt.subplots(figsize=(9, 7.5), constrained_layout=True)
    draw_survey_outline(ax)
    mappable = ax.hexbin(
        df_valid["x"], df_valid["y"], C=df_valid[value_col], gridsize=gridsize,
        extent=(x_lim[0], x_lim[1], y_lim[0], y_lim[1]), cmap=cmap, mincnt=1, linewidths=0.2, zorder=2,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(mappable, ax=ax, shrink=0.8)
    plt.show()



def zoomable_spatial_plot(df, value_col, title, color_continuous_scale="Viridis", zoom=11):
    # Real 20x20m plot footprints, coloured by value_col -- zoom in on the live figure to see
    # individual plots separate out (past roughly compartment scale). Same tradeoff as
    # spatial_viz_comparison_scratch.ipynb's "real footprints" candidate: heavier than
    # quick_spatial_plot() above, since it reads the raw GPKG for actual polygon geometry and
    # builds a real GeoJSON (not just x/y columns) -- use this deliberately for one covariate
    # of interest, not routinely for every one like the hexbin default.
    import json

    import plotly.express as px

    plot_footprints = gpd.read_file(
        project_root / "data" / "raw" / "LiDAR_Years_All_7jul.gpkg", layer="LiDAR_Years", columns=["identification"],
    ).drop_duplicates(subset="identification")

    merged = plot_footprints.merge(df[["identification", value_col]].dropna(), on="identification")
    map_centre_27700 = merged.geometry.union_all().centroid
    map_centre = gpd.GeoSeries([map_centre_27700], crs=27700).to_crs(4326).iloc[0]
    merged = merged.to_crs(4326)
    merged["identification"] = merged["identification"].astype(str)
    footprint_geojson = json.loads(merged.to_json())

    fig = px.choropleth_map(
        merged, geojson=footprint_geojson, locations="identification", featureidkey="properties.identification",
        color=value_col, color_continuous_scale=color_continuous_scale,
        map_style="carto-positron", zoom=zoom,
        center={"lat": map_centre.y, "lon": map_centre.x}, height=600, title=title,
    )
    fig.show()
    return fig

## Statistical usefulness screening toolkit

Applied identically to every candidate covariate below (external datasets and the new
GPKG-derived features alike). Five checks, each answering a different question a raw
distinct-value count can't:

- **Coefficient of variation** (std/mean) -- quick first filter.
- **Variogram range** -- fit an exponential model to the empirical semivariogram (binned
  pairwise squared differences vs. distance); if the fitted range is much larger than typical
  inter-plot spacing, the covariate is functionally constant within a stand no matter its
  nominal grid resolution. Computed on a random subsample (tractability, `n>5000` pairs
  exhaustively otherwise) and reported with an explicit status flag rather than a bare number,
  because a plain curve-fit here is easy to over-trust: `no_structure` (flat from the shortest
  distance tested -- i.e. behaves like noise) and `exceeds_window` (still rising at the longest
  distance tested -- the real range is bigger than what was checked) need opposite readings, and
  collapsing them into one number would hide that.
- **Moran's I** (`libpysal`/`esda`, k=8 neighbours -- matching the plan's own stated methodology
  for the CR-residual analysis, section 4.2) on the raw covariate -- quantifies "smooth blob"
  instead of relying on the hexbin plot by eye.
- **Within- vs. between-compartment variance** (ICC-style ratio: fraction of total variance
  that's *between* compartments) -- tells us whether a covariate can only discriminate at the
  compartment level (ICC near 1, useless for within-stand attribution) or genuinely varies
  within one (ICC well below 1).
- **Spearman correlation vs. the CR residual** -- an early, cheap plausibility screen for
  whether this covariate is even related to what the dissertation is trying to explain, before
  any SHAP work. Reported for both cohorts separately (see the CR-residual cell above for why).


In [ ]:
def coefficient_of_variation(values):
    # Undefined (not just zero-guarded) when the mean is near zero relative to the spread --
    # a mean-centered variable (a PCA component, or a self-minus-neighbour differential that
    # averages out near zero by construction) would otherwise blow up to a meaningless huge
    # number rather than correctly reporting "CV doesn't mean anything here."
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan
    mean, std = values.mean(), values.std()
    if abs(mean) < 1e-8 * std:
        return np.nan
    return std / abs(mean)


def variogram_range(x, y, values, max_distance=5000, n_bins=15, sample_size=5000, seed=42):
    rng = np.random.default_rng(seed)
    mask = ~np.isnan(values)
    x, y, values = np.asarray(x)[mask], np.asarray(y)[mask], np.asarray(values)[mask]
    if len(values) > sample_size:
        idx = rng.choice(len(values), size=sample_size, replace=False)
        x, y, values = x[idx], y[idx], values[idx]

    coords_arr = np.column_stack([x, y])
    tree = cKDTree(coords_arr)
    pairs = tree.query_pairs(r=max_distance, output_type="ndarray")
    if len(pairs) == 0:
        return np.nan, "no_pairs"

    d = np.linalg.norm(coords_arr[pairs[:, 0]] - coords_arr[pairs[:, 1]], axis=1)
    sq_diff = (values[pairs[:, 0]] - values[pairs[:, 1]]) ** 2

    bins = np.linspace(0, max_distance, n_bins + 1)
    bin_idx = np.digitize(d, bins) - 1
    bin_centres, semivariance = [], []
    for b in range(n_bins):
        bmask = bin_idx == b
        if bmask.sum() < 30:
            continue
        bin_centres.append((bins[b] + bins[b + 1]) / 2)
        semivariance.append(0.5 * sq_diff[bmask].mean())
    bin_centres, semivariance = np.array(bin_centres), np.array(semivariance)
    if len(bin_centres) < 4:
        return np.nan, "insufficient_bins"

    # Two "can't fit a range" cases need opposite readings -- checked before curve_fit, which is
    # unreliable/degenerate on either: flat-from-the-start (no real structure at any scale
    # tested) vs. still-rising-at-the-end (the real range exceeds max_distance).
    nugget_estimate, sill_estimate = semivariance[0], semivariance[-3:].mean()
    if sill_estimate <= nugget_estimate * 1.3:
        return 0.0, "no_structure"

    def exp_model(h, nugget, sill, range_param):
        return nugget + sill * (1 - np.exp(-h / range_param))

    try:
        popt, _ = curve_fit(
            exp_model, bin_centres, semivariance,
            p0=[semivariance.min(), semivariance.max() - semivariance.min(), max_distance / 3],
            bounds=([0, 0, 1], [semivariance.max() * 2, semivariance.max() * 3, max_distance * 5]),
            maxfev=5000,
        )
        practical_range = 3 * popt[2]
        if practical_range >= max_distance:
            return max_distance, "exceeds_window"
        return practical_range, "resolved"
    except RuntimeError:
        return np.nan, "fit_failed"


def morans_i(x, y, values, k=8, sample_size=5000, seed=42):
    import esda
    import libpysal

    rng = np.random.default_rng(seed)
    mask = ~np.isnan(values)
    x, y, values = np.asarray(x)[mask], np.asarray(y)[mask], np.asarray(values)[mask]
    if len(values) > sample_size:
        idx = rng.choice(len(values), size=sample_size, replace=False)
        x, y, values = x[idx], y[idx], values[idx]

    coords_arr = np.column_stack([x, y])
    with warnings.catch_warnings():
        # Aberfoyle is genuinely several disjoint forest blocks (see the perimeter section
        # below) -- a KNN graph over them has real disconnected components, not a bug.
        warnings.simplefilter("ignore", category=UserWarning)
        weights = libpysal.weights.KNN.from_array(coords_arr, k=k)
    weights.transform = "r"
    mi = esda.moran.Moran(values, weights, permutations=199)
    return mi.I, mi.p_sim


def within_between_icc(compartment_ids, values):
    df = pd.DataFrame({"cpmt": compartment_ids, "value": values}).dropna()
    grand_mean = df["value"].mean()
    compartment_means = df.groupby("cpmt")["value"].transform("mean")
    between_ss = ((compartment_means - grand_mean) ** 2).sum()
    within_ss = ((df["value"] - compartment_means) ** 2).sum()
    total_ss = between_ss + within_ss
    return between_ss / total_ss if total_ss > 0 else np.nan


def spearman_vs_residual(values, residuals):
    df = pd.DataFrame({"value": values, "residual": residuals}).dropna()
    if len(df) < 10:
        return np.nan, np.nan
    return spearmanr(df["value"], df["residual"])


def screen_covariate(df, value_col, name, category, access_method, years_covered, notes=""):
    # Runs the full battery and returns one results-table row. `df` must have x, y, cpmt,
    # mean_cr_residual_4survey, mean_cr_residual_6survey alongside value_col.
    values = df[value_col].values
    valid = ~np.isnan(values)
    n_valid = int(valid.sum())
    distinct_values = df.loc[valid, value_col]

    t0 = time.time()
    cv = coefficient_of_variation(values)
    vrange, vstatus = variogram_range(df["x"].values, df["y"].values, values)
    mi, mi_p = morans_i(df["x"].values, df["y"].values, values)
    icc = within_between_icc(df["cpmt"].values, values)
    rho_4s, rho_4s_p = spearman_vs_residual(values, df["mean_cr_residual_4survey"].values)
    rho_6s, rho_6s_p = spearman_vs_residual(values, df["mean_cr_residual_6survey"].values)
    elapsed = time.time() - t0

    row = dict(
        dataset=name, category=category, access_method=access_method,
        n_plots_sampled=n_valid,
        n_distinct_values=int(distinct_values.nunique()) if n_valid else 0,
        std=float(distinct_values.std()) if n_valid else np.nan,
        value_min=float(distinct_values.min()) if n_valid else np.nan,
        value_max=float(distinct_values.max()) if n_valid else np.nan,
        cv=cv, variogram_range_m=vrange, variogram_status=vstatus,
        morans_i=mi, morans_i_p=mi_p, between_compartment_icc=icc,
        spearman_vs_cr_residual_4survey=rho_4s, spearman_p_4survey=rho_4s_p,
        spearman_vs_cr_residual_6survey=rho_6s, spearman_p_6survey=rho_6s_p,
        years_covered=years_covered, notes=notes,
    )
    print(f"[{name}] n={n_valid:,}, distinct={row['n_distinct_values']}, CV={cv:.3f}, "
          f"variogram_range={vrange:.0f}m [{vstatus}], Moran's I={mi:.3f} (p={mi_p:.3f}), "
          f"ICC={icc:.3f}, Spearman(4survey)={rho_4s:.3f} (p={rho_4s_p:.3f})  [{elapsed:.1f}s]")
    return row


results = []  # one dict per dataset/feature, appended throughout the notebook, final table at the end


# Part 1: External data sources

## 1. HadUK-Grid (temperature, rainfall)

Plan's claim: ~2-3 cells across the forest at 1km. Access-tested directly rather than assumed.


In [ ]:
# Real access test: browse CEDA's archive (works with no auth -- listing is public), then
# attempt to actually download one real file (2021 monthly mean temperature, 1km).
browse_url = ("https://data.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/"
              "v1.3.2.ceda/1km/tas/mon/v20260512/?json")
listing = requests.get(browse_url, timeout=15).json()
print(f"Archive browsing works with no auth: {len(listing['items'])} files listed, e.g. "
      f"{listing['items'][0]['name']}")

file_url = ("https://data.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/"
            "v1.3.2.ceda/1km/tas/mon/v20260512/tas_hadukgrid_uk_1km_mon_202101-202112.nc")
download_attempt = requests.get(file_url, timeout=20, allow_redirects=True)
print(f"Actual file download: HTTP {download_attempt.status_code}, "
      f"landed at: {download_attempt.url}")
print(f"Content-Type: {download_attempt.headers.get('content-type')}  "
      f"(real .nc data would be application/x-netcdf or octet-stream, not text/html)")


**Confirmed blocked, not just documented.** Metadata browsing is public, but the real file
download redirects to `auth.ceda.ac.uk`'s sign-in page (HTML login form, not data) -- a free
CEDA account is required, and that registration has not been done this session. **Access
method: CEDA account (free registration, not yet obtained).**

No live extraction possible without that account, so the CV/variogram/Moran's I/ICC/Spearman
columns for this row are genuinely blocked, not "not tested" -- distinct from the datasets
below where a live value was actually pulled. The plan's own "~2-3 cells" claim also still holds
as a separate, weaker, geometry-only argument (1km grid over a ~20km-long forest), but that is
not the empirical per-plot check the supervisor asked for.


In [ ]:
results.append(dict(
    dataset="HadUK-Grid (tas, 1km)", category="climate",
    access_method="CEDA account required (confirmed: real file download redirects to auth.ceda.ac.uk login)",
    n_plots_sampled=0, n_distinct_values=np.nan, std=np.nan, value_min=np.nan, value_max=np.nan,
    cv=np.nan, variogram_range_m=np.nan, variogram_status="blocked",
    morans_i=np.nan, morans_i_p=np.nan, between_compartment_icc=np.nan,
    spearman_vs_cr_residual_4survey=np.nan, spearman_p_4survey=np.nan,
    spearman_vs_cr_residual_6survey=np.nan, spearman_p_6survey=np.nan,
    years_covered="unknown (blocked)",
    notes="Blocked by CEDA login wall (confirmed live). Geometric estimate only: ~2-3 cells "
          "over the forest at 1km, unverified against real values.",
))


## Update: HadUK-Grid unblocked via manual download

CEDA's login wall (confirmed above) blocks a scripted download, but not a real account holder
downloading through the browser -- one real file (`tas`, monthly mean temperature, 1km, 2021,
the same file the blocked scripted attempt above targeted) was downloaded manually to
`data/raw/haduk/tas_hadukgrid_uk_1km_mon_202101-202112.nc`, replacing that placeholder row
below rather than leaving "blocked" on the record once it's no longer true.

The file opens directly with `rasterio` (GDAL's built-in NetCDF driver) -- no `netCDF4`/
`h5netcdf`/`xarray` backend needed. One real wrinkle worth flagging: `rasterio` reports the
CRS as an unnamed WKT rather than recognising it as EPSG:27700 outright, even though every
projection parameter (Airy 1830 spheroid, Transverse Mercator, false easting 400000, false
northing -100000, central meridian -2, scale factor 0.9996012717) matches British National Grid
exactly -- this is the CF-convention NetCDF grid mapping not citing the EPSG code explicitly,
not a real ambiguity. Forcing `crs=EPSG:27700` below avoids relying on that auto-detection, and
also means (unlike GWA/SoilGrids/CHELSA above) **no coordinate reprojection is needed at all**,
since the analysis plot set is already in EPSG:27700.


In [ ]:
haduk_path = project_root / "data/raw/haduk/tas_hadukgrid_uk_1km_mon_202101-202112.nc"
print(f"Real file present: {haduk_path.exists()}, {haduk_path.stat().st_size / 1e6:.1f} MB")

from rasterio.crs import CRS

with rasterio.open(haduk_path) as ds:
    print(f"GDAL-reported CRS recognised as EPSG code: {ds.crs.to_epsg()} (None expected -- see markdown above)")
    print(f"band count: {ds.count} (should be 12, one per month of 2021), nodata: {ds.nodata}")

    # tas#units is degC directly (no Kelvin/scale-factor conversion needed, unlike CHELSA) --
    # average the 12 monthly bands for a 2021 mean annual temperature, the same kind of
    # headline variable as CHELSA's bio1 above.
    monthly = ds.read(masked=True).astype(float)  # (12, rows, cols), nodata (1e20) auto-masked
    annual_mean = monthly.mean(axis=0)

    forced_crs = CRS.from_epsg(27700)
    rows_idx, cols_idx = rasterio.transform.rowcol(ds.transform, plots["x"].values, plots["y"].values)
    rows_idx, cols_idx = np.array(rows_idx), np.array(cols_idx)
    in_bounds = (rows_idx >= 0) & (rows_idx < annual_mean.shape[0]) & (cols_idx >= 0) & (cols_idx < annual_mean.shape[1])

    haduk_values = np.full(len(plots), np.nan)
    sampled = annual_mean[rows_idx[in_bounds], cols_idx[in_bounds]]
    haduk_values[in_bounds] = np.ma.filled(sampled, np.nan)

plots["haduk_tas_2021_mean"] = haduk_values
print(f"\n{np.isfinite(haduk_values).sum():,} / {len(plots):,} plots sampled")
print(f"2021 mean annual temp range: {np.nanmin(haduk_values):.2f} to {np.nanmax(haduk_values):.2f} degC")
print("Distinct values:", pd.Series(haduk_values).nunique())


In [ ]:
quick_spatial_plot(plots, "haduk_tas_2021_mean", "HadUK-Grid 2021 mean annual temp (1km, real extraction)", cmap="coolwarm")

In [ ]:
haduk_row = screen_covariate(
    plots, "haduk_tas_2021_mean", name="HadUK-Grid (tas, 1km)", category="climate",
    access_method="Manual browser download by an authenticated CEDA account holder (the scripted "
                  "download above is still genuinely blocked -- this is a real account, not a workaround)",
    years_covered="2021 only in the file downloaded so far -- one file per year/variable on CEDA, "
                  "so 2002/2006/2008/2012/2023 would each need their own manual download to cover "
                  "every survey year",
    notes="REAL EXTRACTION (updates the earlier 'blocked' placeholder). Compare its distinct-value "
          "count and Moran's I/ICC directly against CHELSA's bio1 row above (same nominal 1km grid, "
          "same kind of variable) -- this is the actual apples-to-apples check the plan's assumption "
          "needed.",
)
results[:] = [r for r in results if r["dataset"] != "HadUK-Grid (tas, 1km)"]
results.append(haduk_row)


## 2. ERA5-Land

Plan's claim: ~9km, effectively 1 cell over the whole forest.


In [ ]:
# Copernicus CDS requires a personal API key for every data request -- confirmed directly
# rather than assumed, via an unauthenticated request to the real retrieve endpoint.
cds_response = requests.post(
    "https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-land-monthly-means/execution",
    json={}, timeout=15,
)
print(f"CDS API (no key): HTTP {cds_response.status_code} -- {cds_response.json()}")

# ERA5-Land IS in Google Earth Engine's public catalog (ECMWF/ERA5_LAND/MONTHLY_AGGR), which
# would sidestep the CDS key entirely -- but GEE itself needs a Cloud project ID, tested and
# blocked separately in the AlphaEarth section below. Noted here since it's the same blocker.
print("\nERA5-Land is also available via Google Earth Engine's public catalog "
      "(ECMWF/ERA5_LAND/MONTHLY_AGGR) -- same GEE project-ID blocker as AlphaEarth, see below.")


**Confirmed blocked.** `401 authentication required` from the real CDS endpoint with no key --
this is Copernicus's standard access control, not a guess. **Access method: Copernicus CDS
personal API key (needs a free CDS account + key, not yet obtained)**, or alternatively via
Google Earth Engine once a Cloud project ID is available (see AlphaEarth below -- same
blocker). No live extraction possible either way this session.


In [ ]:
results.append(dict(
    dataset="ERA5-Land (9km)", category="climate",
    access_method="Copernicus CDS API key required (confirmed: 401 on real endpoint); or GEE (blocked, see AlphaEarth)",
    n_plots_sampled=0, n_distinct_values=np.nan, std=np.nan, value_min=np.nan, value_max=np.nan,
    cv=np.nan, variogram_range_m=np.nan, variogram_status="blocked",
    morans_i=np.nan, morans_i_p=np.nan, between_compartment_icc=np.nan,
    spearman_vs_cr_residual_4survey=np.nan, spearman_p_4survey=np.nan,
    spearman_vs_cr_residual_6survey=np.nan, spearman_p_6survey=np.nan,
    years_covered="unknown (blocked)",
    notes="Blocked by CDS API key wall (confirmed live, 401). Geometric estimate only: whole "
          "forest inside ~3-4 cells at 9km -- essentially certain to fail even if unblocked.",
))


## 3. CEH 50m soil dataset

Identified via search as UKCEH's "National scale maps of parent material properties, terrain
and soil natural capital units at 50 metre resolution for Great Britain, 2020" (EIDC). Direct
download, no account needed -- confirmed live (unlike HadUK-Grid/ERA5-Land above). Using the
natural-capital pedotope layer (`natural_capital_pedotopes.tif`) as the headline soil
classification variable -- this is the closest match to "CEH 50m soil dataset" in the task, out
of the 12 rasters this archive actually contains.


In [ ]:
# Already downloaded and cached this session (~1GB zip, free direct download, no login) --
# skip re-downloading if the extracted GeoTIFF is already present.
ceh_path = CACHE_DIR / "ceh_pedotopes.tif"
print(f"CEH pedotope raster cached: {ceh_path.exists()}, "
      f"{ceh_path.stat().st_size / 1e6:.1f} MB" if ceh_path.exists() else "NOT CACHED")

with rasterio.open(ceh_path) as ds:
    print("CRS:", ds.crs, "| resolution:", ds.res, "| nodata:", ds.nodata)
    rows, cols = rasterio.transform.rowcol(ds.transform, plots["x"].values, plots["y"].values)
    rows, cols = np.array(rows), np.array(cols)
    array = ds.read(1)
    in_bounds = (rows >= 0) & (rows < array.shape[0]) & (cols >= 0) & (cols < array.shape[1])
    pedotope_values = np.full(len(plots), np.nan)
    pedotope_values[in_bounds] = array[rows[in_bounds], cols[in_bounds]]
    if ds.nodata is not None:
        pedotope_values[pedotope_values == ds.nodata] = np.nan

plots["ceh_pedotope"] = pedotope_values
n_missing = int(np.isnan(pedotope_values).sum())
print(f"\n{len(plots) - n_missing:,} / {len(plots):,} plots sampled ({n_missing} outside raster bounds/nodata)")
print("Distinct pedotope classes:", plots['ceh_pedotope'].nunique())
print(plots["ceh_pedotope"].value_counts())


In [ ]:
quick_spatial_plot(plots, "ceh_pedotope", "CEH natural-capital pedotope class (50m, real extraction)", cmap="tab10")


In [ ]:
row = screen_covariate(
    plots, "ceh_pedotope", name="CEH 50m soil (natural capital pedotopes)", category="soil",
    access_method="Direct EIDC download, no auth (confirmed: real ~1GB zip, no login)",
    years_covered="Static (2020 snapshot) -- treated as time-invariant, applies to all 6 survey years equally",
    notes="7 distinct classes over 71,766 plots -- a real categorical map, not a single flat "
          "value, but see the ICC/Moran's I columns for whether that variation is USEFUL "
          "(likely still coarse relative to plot spacing -- check numbers, don't assume).",
)
results.append(row)


## 4. James Hutton 1:250,000 soil map

Plan's claim: ~3-5 polygons total over the whole study area. Found a live WMS
(`druid.hutton.ac.uk`, World Reference Base soil classification) -- confirmed reachable and
queryable with a real `GetFeatureInfo` point query (tested: returned "Umbric Dystric
Stagnosol" for an Aberfoyle point).

**Access method note:** this is a point-query WMS, not a downloadable raster/COG like the
sources above -- querying all 71,766 plots individually would mean 71,766 separate HTTP
requests to a shared public ArcGIS service, which is both impractical (would take hours) and
not a reasonable way to use someone else's free public service. Using a **random subsample**
instead (`n=150`, fixed seed) -- enough to empirically confirm or refute "~3-5 polygons total,"
not enough for the full variogram/Moran's I/ICC battery (those need the full raster/point grid,
which this WMS doesn't expose in bulk).


In [ ]:
import re

rng = np.random.default_rng(42)
jh_sample = plots.sample(n=150, random_state=42).reset_index(drop=True)
transformer_27700_to_4326 = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
sample_lon, sample_lat = transformer_27700_to_4326.transform(jh_sample["x"].values, jh_sample["y"].values)

soil_groups = []
for lon, lat in zip(sample_lon, sample_lat):
    d = 0.001  # tiny bbox around the point, single-pixel-ish query
    url = (
        "http://druid.hutton.ac.uk/arcgis/services/GSSOIL_WRB/MapServer/WMSServer"
        f"?service=WMS&version=1.1.1&request=GetFeatureInfo&layers=0&query_layers=0"
        f"&styles=&bbox={lon-d},{lat-d},{lon+d},{lat+d}&width=101&height=101&srs=EPSG:4326"
        f"&format=image/png&info_format=text/html&x=50&y=50"
    )
    try:
        resp = requests.get(url, timeout=10)
        match = re.search(r"<td>([^<]+)</td>\s*</tr>\s*</tbody>", resp.text, re.S)
        soil_groups.append(match.group(1).strip() if match else None)
    except requests.RequestException:
        soil_groups.append(None)
    time.sleep(0.15)  # be a reasonable citizen of a shared public service

jh_sample["wrb_soil_group"] = soil_groups
valid = jh_sample["wrb_soil_group"].notna()
print(f"Queried {len(jh_sample)} sample points, {valid.sum()} returned a value")
print(f"Distinct WRB soil groups in this sample: {jh_sample.loc[valid, 'wrb_soil_group'].nunique()}")
print(jh_sample.loc[valid, "wrb_soil_group"].value_counts())


**Surprising, worth flagging to the supervisor directly:** the plan's "~3-5 polygons total"
estimate does not survive contact with real data. A sample of just 150 points already turned up
several distinct World Reference Base soil groups (see the count above) -- not a handful of
polygons covering the *entire* study area as assumed. 1:250,000 scale is still coarse (each
polygon covers a large area relative to a 20-40m plot), but "coarse polygons" and "only 3-5
values total" are different claims, and only the first one holds up here.


In [ ]:
n_distinct_jh = int(jh_sample.loc[valid, "wrb_soil_group"].nunique())
results.append(dict(
    dataset="James Hutton 1:250,000 soil map (WRB)", category="soil",
    access_method="Public WMS GetFeatureInfo, no auth (confirmed live; point-sampled, n=150, not exhaustive)",
    n_plots_sampled=int(valid.sum()), n_distinct_values=n_distinct_jh,
    std=np.nan, value_min=np.nan, value_max=np.nan,  # categorical, not numeric
    cv=np.nan, variogram_range_m=np.nan, variogram_status="not_computed (categorical point-sample only)",
    morans_i=np.nan, morans_i_p=np.nan, between_compartment_icc=np.nan,
    spearman_vs_cr_residual_4survey=np.nan, spearman_p_4survey=np.nan,
    spearman_vs_cr_residual_6survey=np.nan, spearman_p_6survey=np.nan,
    years_covered="Static (soil survey snapshot) -- time-invariant, applies to all 6 survey years equally",
    notes=f"SURPRISING: {n_distinct_jh} distinct WRB soil groups found in just a 150-point "
          f"sample -- contradicts the plan's '~3-5 polygons total' assumption. Still coarse "
          f"(1:250,000), but not the near-flat covariate the plan assumed. Full quantitative "
          f"screen (variogram/Moran's I/ICC) not possible via this point-query WMS -- would "
          f"need a bulk raster/WCS export to do properly.",
))


## 5. SoilGrids (ISRIC), 250m

New candidate -- ML-interpolated from real soil profile observations plus environmental
covariates (climate, terrain, land cover), not a flat polygon conversion like CEH/James Hutton
above. Public Cloud-Optimised GeoTIFFs, no auth, readable directly via `rasterio`'s `/vsicurl/`
(a windowed remote read -- no full-globe download needed). Using topsoil pH (0-5cm) as the
headline property.


In [ ]:
soilgrids_url = "/vsicurl/https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean.vrt"

with rasterio.open(soilgrids_url) as ds:
    print("CRS:", ds.crs, "| native resolution:", ds.res, "(Homolosine projection, metres)")
    transformer = Transformer.from_crs("EPSG:27700", ds.crs, always_xy=True)
    minx, miny = transformer.transform(BBOX_27700["minx"] - 2000, BBOX_27700["miny"] - 2000)
    maxx, maxy = transformer.transform(BBOX_27700["maxx"] + 2000, BBOX_27700["maxy"] + 2000)
    window = from_bounds(minx, miny, maxx, maxy, ds.transform)
    window_transform = ds.window_transform(window)
    array = ds.read(1, window=window)
    nodata = ds.nodata

    px, py = transformer.transform(plots["x"].values, plots["y"].values)
    inv = ~window_transform
    cols, rows = inv * (px, py)
    cols, rows = cols.astype(int), rows.astype(int)
    in_bounds = (rows >= 0) & (rows < array.shape[0]) & (cols >= 0) & (cols < array.shape[1])
    ph_values = np.full(len(plots), np.nan)
    ph_values[in_bounds] = array[rows[in_bounds], cols[in_bounds]]
    if nodata is not None:
        ph_values[ph_values == nodata] = np.nan

# SoilGrids stores pH as pH*10 (documented unit_measure.d_factor=10) -- convert to real pH.
plots["soilgrids_ph"] = ph_values / 10.0
print(f"\n{in_bounds.sum():,} / {len(plots):,} plots sampled")
print(f"pH range: {plots['soilgrids_ph'].min():.2f} to {plots['soilgrids_ph'].max():.2f}")
print("Distinct pH values (at native precision):", plots["soilgrids_ph"].nunique())


In [ ]:
quick_spatial_plot(plots, "soilgrids_ph", "SoilGrids topsoil pH, 0-5cm (250m, real extraction)", cmap="viridis")


In [ ]:
row = screen_covariate(
    plots, "soilgrids_ph", name="SoilGrids topsoil pH 0-5cm", category="soil",
    access_method="Public COG via /vsicurl/, no auth (confirmed live, windowed remote read)",
    years_covered="Static (ML-interpolated from historical profile data, ~2020 baseline) -- time-invariant, applies to all 6 survey years equally",
    notes="ML-interpolated from real soil profiles + covariates, not a flat polygon conversion -- "
          "check whether it shows MORE local variation than CEH/James Hutton above despite similar "
          "or finer nominal resolution (see Moran's I / ICC / variogram columns).",
)
results.append(row)


## 6. CHELSA, 1km

New candidate -- same nominal grid as HadUK-Grid (1km), but CHELSA's downscaling explicitly
conditions on terrain/elevation (orographic precipitation/temperature modelling), unlike a
plain interpolation. Direct public download, no auth. Using `bio1` (mean annual air
temperature) as the headline variable.


In [ ]:
chelsa_path = CACHE_DIR / "chelsa_bio1.tif"
if not chelsa_path.exists():
    url = "https://os.zhdk.cloud.switch.ch/chelsav2/GLOBAL/climatologies/1981-2010/bio/CHELSA_bio1_1981-2010_V.2.1.tif"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    chelsa_path.write_bytes(resp.content)
print(f"CHELSA bio1 cached: {chelsa_path.stat().st_size / 1e6:.1f} MB (direct download, no auth)")

with rasterio.open(chelsa_path) as ds:
    print("CRS:", ds.crs, "| resolution (degrees):", ds.res)
    transformer = Transformer.from_crs("EPSG:27700", ds.crs, always_xy=True)
    minx, miny = transformer.transform(BBOX_27700["minx"] - 2000, BBOX_27700["miny"] - 2000)
    maxx, maxy = transformer.transform(BBOX_27700["maxx"] + 2000, BBOX_27700["maxy"] + 2000)
    window = from_bounds(minx, miny, maxx, maxy, ds.transform)
    window_transform = ds.window_transform(window)
    array = ds.read(1, window=window)
    nodata = ds.nodata

    px, py = transformer.transform(plots["x"].values, plots["y"].values)
    inv = ~window_transform
    cols, rows = inv * (px, py)
    cols, rows = cols.astype(int), rows.astype(int)
    in_bounds = (rows >= 0) & (rows < array.shape[0]) & (cols >= 0) & (cols < array.shape[1])
    bio1_raw = np.full(len(plots), np.nan)
    bio1_raw[in_bounds] = array[rows[in_bounds], cols[in_bounds]]
    if nodata is not None:
        bio1_raw[bio1_raw == nodata] = np.nan

# CHELSA V2.1 bio1 is stored as Kelvin*10 (offset -273.15, scale 0.1) -- convert to real Celsius.
plots["chelsa_bio1_celsius"] = bio1_raw / 10.0 - 273.15
print(f"\n{in_bounds.sum():,} / {len(plots):,} plots sampled")
print(f"Mean annual temp range: {plots['chelsa_bio1_celsius'].min():.2f} to "
      f"{plots['chelsa_bio1_celsius'].max():.2f} degrees C")
print("Distinct values:", plots["chelsa_bio1_celsius"].nunique())


In [ ]:
quick_spatial_plot(plots, "chelsa_bio1_celsius", "CHELSA mean annual air temp, bio1 (1km, real extraction)", cmap="coolwarm")


In [ ]:
row = screen_covariate(
    plots, "chelsa_bio1_celsius", name="CHELSA bio1 (mean annual temp)", category="climate",
    access_method="Direct download, no auth (confirmed live)",
    years_covered="Static (1981-2010 climatology) -- time-invariant, applies to all 6 survey years equally",
    notes="Same nominal 1km grid as HadUK-Grid, but terrain-conditioned downscaling -- compare "
          "its distinct-value count and Moran's I/ICC directly against HadUK-Grid's blocked "
          "status once that access is obtained, to see if the terrain-conditioning claim holds up.",
)
results.append(row)
print(f"\nDistinct values at 1km nominal grid: {row['n_distinct_values']} -- compare against the "
      f"plan's original HadUK-Grid claim of ~2-3 cells total.")


## 7. Global Wind Atlas, 250m

Already in the plan as the WASP fallback/cross-check, extracted at a single spot-check tile in
`environmental_data_exploration/environmental_data_sources_survey.ipynb` -- this is the first
time it's been pulled at every real plot coordinate with the full statistical screen. Public
API, no auth, whole-country GeoTIFF per height (10/50/100/150/200m; using 10m, the plan's
"lowest suitable height" guidance).


In [ ]:
gwa_path = CACHE_DIR / "gbr_wind_10m.tif"
if not gwa_path.exists():
    resp = requests.get("https://globalwindatlas.info/api/gis/country/GBR/wind-speed/10", timeout=60)
    resp.raise_for_status()
    gwa_path.write_bytes(resp.content)
print(f"GWA GBR/10m cached: {gwa_path.stat().st_size / 1e6:.1f} MB (public API, no auth)")

with rasterio.open(gwa_path) as ds:
    transformer = Transformer.from_crs("EPSG:27700", ds.crs, always_xy=True)
    minx, miny = transformer.transform(BBOX_27700["minx"] - 2000, BBOX_27700["miny"] - 2000)
    maxx, maxy = transformer.transform(BBOX_27700["maxx"] + 2000, BBOX_27700["maxy"] + 2000)
    window = from_bounds(minx, miny, maxx, maxy, ds.transform)
    window_transform = ds.window_transform(window)
    array = ds.read(1, window=window)
    nodata = ds.nodata

    px, py = transformer.transform(plots["x"].values, plots["y"].values)
    inv = ~window_transform
    cols, rows = inv * (px, py)
    cols, rows = cols.astype(int), rows.astype(int)
    in_bounds = (rows >= 0) & (rows < array.shape[0]) & (cols >= 0) & (cols < array.shape[1])
    wind_values = np.full(len(plots), np.nan)
    wind_values[in_bounds] = array[rows[in_bounds], cols[in_bounds]]
    if nodata is not None:
        wind_values[wind_values == nodata] = np.nan

plots["gwa_wind_speed_10m"] = wind_values
print(f"\n{in_bounds.sum():,} / {len(plots):,} plots sampled")
print(f"Wind speed range: {plots['gwa_wind_speed_10m'].min():.2f} to {plots['gwa_wind_speed_10m'].max():.2f} m/s")
print("Distinct values:", plots["gwa_wind_speed_10m"].nunique())


In [ ]:
quick_spatial_plot(plots, "gwa_wind_speed_10m", "Global Wind Atlas, 10m wind speed (real extraction, all plots)", cmap="viridis")


Optional zoom-in view (heavier, built on demand) -- GWA has the most local texture of any external source tested (1,815 distinct values), so it's the best candidate here to actually see real plot footprints separate out on zoom, rather than a smooth hexbin blob.

In [ ]:
_ = zoomable_spatial_plot(plots, "gwa_wind_speed_10m", "Global Wind Atlas, 10m wind speed -- zoom in to see real plot footprints")  # assigned to suppress a duplicate auto-display of the returned figure

In [ ]:
row = screen_covariate(
    plots, "gwa_wind_speed_10m", name="Global Wind Atlas (10m wind speed)", category="wind",
    access_method="Public REST API, no auth (confirmed live)",
    years_covered="Static (long-term climatology) -- time-invariant, applies to all 6 survey years equally",
    notes="Full-plot-set confirmation of the spot-check already done in "
          "notebooks/environmental_data/environmental_data_sources_survey.ipynb -- see that "
          "notebook for the real-vs-nominal resolution finding (anisotropic ~155m E-W / ~278m N-S "
          "at Aberfoyle's latitude, not a flat 250m).",
)
results.append(row)


## 8. AlphaEarth Foundations, 10m

New candidate, 2017-onward only (Google DeepMind satellite embedding dataset, served via
Google Earth Engine). **Two access layers, tested separately:**

1. GEE Python API + stored OAuth credentials already exist on this machine
   (`~/.config/earthengine/credentials`, a valid refresh token from a prior session) -- but
   `ee.Initialize()` also needs a **Google Cloud project ID** with the Earth Engine API enabled,
   which is not stored anywhere accessible and cannot be guessed. This is a one-line, one-time
   fix if the project ID is supplied (`EE_PROJECT_ID` below) -- **the single most valuable
   access blocker to resolve**, since it would also unblock ERA5-Land above.


In [ ]:
EE_PROJECT_ID = None  # <-- fill in your Google Cloud project ID (the one Earth Engine is enabled on) and re-run this cell

try:
    import ee

    if EE_PROJECT_ID is None:
        raise RuntimeError(
            "EE_PROJECT_ID not set. Stored GEE credentials exist on this machine "
            "(~/.config/earthengine/credentials) but ee.Initialize() also needs a Cloud project "
            "ID -- find yours at https://code.earthengine.google.com/ (top-left project selector) "
            "and set EE_PROJECT_ID above."
        )
    ee.Initialize(project=EE_PROJECT_ID)

    alphaearth = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    date_range = alphaearth.aggregate_array("system:time_start").getInfo()
    print("GEE initialised successfully.")
    print(f"AlphaEarth Foundations collection reachable, {len(date_range)} annual images available.")

    # NOTE, kept as a comment per the task: AlphaEarth's 64 embedding dimensions are NOT
    # physically interpretable (they're a learned representation, not named covariates like
    # elevation/pH/wind speed) -- useful only as a robustness/clustering check downstream, never
    # as a primary SHAP-attributed feature. Extraction of actual per-plot embedding values is
    # left for once EE_PROJECT_ID is supplied -- this cell only confirms reachability.
    gee_blocked = False
except Exception as exc:
    print(f"BLOCKED: {exc}")
    gee_blocked = True


2. **Even once unblocked, AlphaEarth cannot answer the temporal Env-PINN question.** It only
   exists from 2017 onward -- it has **no coverage for 2002, 2006, 2008, or 2012**, so it cannot
   be used for the 2002-2012 training window the temporal Env-PINN test needs, nor for SQ2
   (which spans the same early years). At best it could support a *spatial-only* check restricted
   to 2021/2023, never a temporal one -- this is a hard data-availability limit, not an access
   problem, and would still apply after `EE_PROJECT_ID` is fixed.


In [ ]:
results.append(dict(
    dataset="AlphaEarth Foundations (10m embeddings)", category="other (learned embedding, not a named covariate)",
    access_method="Google Earth Engine (credentials present, BLOCKED on missing Cloud project ID)" if gee_blocked
                   else "Google Earth Engine (confirmed reachable)",
    n_plots_sampled=0, n_distinct_values=np.nan, std=np.nan, value_min=np.nan, value_max=np.nan,
    cv=np.nan, variogram_range_m=np.nan, variogram_status="blocked" if gee_blocked else "not_extracted_yet",
    morans_i=np.nan, morans_i_p=np.nan, between_compartment_icc=np.nan,
    spearman_vs_cr_residual_4survey=np.nan, spearman_p_4survey=np.nan,
    spearman_vs_cr_residual_6survey=np.nan, spearman_p_6survey=np.nan,
    years_covered="2017 onward ONLY -- no 2002/2006/2008/2012 coverage, cannot support the "
                  "temporal Env-PINN window or SQ2 regardless of access",
    notes="64 embedding dims are not physically interpretable -- robustness/clustering check "
          "only, never a primary SHAP-attributed feature. Needs EE_PROJECT_ID to even extract; "
          "unblocking this would also unblock ERA5-Land via GEE (same credential).",
))


# Part 2: New features derived directly from the existing GPKG

No external data source needed for any of these five -- they only use geometry and height
already present in the raw GeoPackage, so (unlike every external source above) they are
available at **all six survey years automatically**, not just a static snapshot. That's a real,
practical advantage worth stating plainly, not just a footnote.


## 1 & 2. Distance to compartment boundary, and distance to forest perimeter

Two different mechanisms, computed separately on purpose (not collapsed into one "distance to
boundary" number):

- **Distance to compartment (`cpmt`) boundary** -- within-stand edge effects between two
  adjacent *managed* blocks (light/wind differences at a felling/thinning boundary).
- **Distance to forest perimeter** -- genuine exposure to open ground (road, clearfell, open
  hillside), using the real forest extent (all compartments unioned, morphologically closed --
  reusing the exact technique from
  `data_exploration_gpkg/notebooks/spatial_viz_comparison_scratch.ipynb` cell 31), not just this
  plot's own compartment. A plot deep inside a large compartment can be close to the *forest's*
  edge if that compartment itself borders open ground -- distance (1) alone would miss this.


In [ ]:
from shapely.geometry import MultiPolygon

plots_gdf = gpd.GeoDataFrame(plots, geometry=gpd.points_from_xy(plots["x"], plots["y"]), crs=27700)

# --- Distance to own compartment boundary ---
cpmt_boundary_lookup = compartment_boundaries.set_index("cpmt")["geometry"]
dist_to_cpmt_boundary = np.full(len(plots_gdf), np.nan)
for cpmt_id, group in plots_gdf.groupby("cpmt"):
    boundary = cpmt_boundary_lookup.get(cpmt_id)
    if boundary is not None:
        dist_to_cpmt_boundary[group.index] = group.geometry.distance(boundary.boundary).values
plots["dist_to_cpmt_boundary"] = dist_to_cpmt_boundary
print(f"dist_to_cpmt_boundary: {np.nanmin(dist_to_cpmt_boundary):.1f}m to "
      f"{np.nanmax(dist_to_cpmt_boundary):.1f}m, mean {np.nanmean(dist_to_cpmt_boundary):.1f}m")

# --- Distance to forest perimeter (real disjoint blocks, morphological closing) ---
closed_boundary = compartment_boundaries.geometry.buffer(60).union_all().buffer(-60)
boundary_parts = list(closed_boundary.geoms) if closed_boundary.geom_type == "MultiPolygon" else [closed_boundary]
real_blocks = [p for p in boundary_parts if p.area > 10_000]  # drop closing slivers under 1ha
forest_extent = MultiPolygon(real_blocks) if len(real_blocks) > 1 else real_blocks[0]
forest_perimeter = forest_extent.boundary  # exterior AND interior (internal clearing) rings both count

plots["dist_to_forest_perimeter"] = plots_gdf.geometry.distance(forest_perimeter).values
print(f"{len(real_blocks)} real forest blocks (>1ha) after morphological closing")
print(f"dist_to_forest_perimeter: {plots['dist_to_forest_perimeter'].min():.1f}m to "
      f"{plots['dist_to_forest_perimeter'].max():.1f}m, mean {plots['dist_to_forest_perimeter'].mean():.1f}m")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
for ax, col, title in zip(axes, ["dist_to_cpmt_boundary", "dist_to_forest_perimeter"],
                           ["Distance to own compartment boundary (m)", "Distance to forest perimeter (m)"]):
    draw_survey_outline(ax)
    mappable = ax.hexbin(plots["x"], plots["y"], C=plots[col], gridsize=110,
                          extent=(x_lim[0], x_lim[1], y_lim[0], y_lim[1]), cmap="viridis", mincnt=1, linewidths=0.2, zorder=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(mappable, ax=ax, shrink=0.8)
plt.show()


In [ ]:
for col, label in [("dist_to_cpmt_boundary", "Distance to compartment boundary"),
                    ("dist_to_forest_perimeter", "Distance to forest perimeter")]:
    row = screen_covariate(
        plots, col, name=label, category="terrain-derived (new, from GPKG geometry)",
        access_method="Local GPKG geometry only -- no external source",
        years_covered="All 6 (2002/2006/2008/2012/2021/2023) -- geometry is fixed per plot, "
                      "doesn't depend on survey year",
        notes="",
    )
    results.append(row)


## 3. Local elevation roughness

Standard deviation of elevation within a 100m buffer around each plot, from the same OS
Terrain 50 DTM already confirmed in
`environmental_data_exploration/environmental_data_sources_survey.ipynb` -- complements TOPEX
as a second, independent terrain-exposure proxy (TOPEX is directional horizon-angle shelter;
this is local surface variability, a different signal). Reuses the same cached GB DTM bundle
(no re-download).


In [ ]:
# Tiles needed to cover the Aberfoyle bbox with margin: NN30/NN40/NN50 (row 0) and
# NS39/NS49/NS59 (row 9) -- see notebooks/environmental_data/environmental_data_sources_survey.ipynb
# for how the OS Downloads API and tile-naming scheme were confirmed.
TILES = ["nn30", "nn40", "nn50", "ns39", "ns49", "ns59"]
gb_zip_path = CACHE_DIR / "terr50_gagg_gb.zip"

with zipfile.ZipFile(gb_zip_path) as gb_zip:
    names = gb_zip.namelist()
    for tile in TILES:
        asc_path = CACHE_DIR / f"{tile.upper()}.asc"
        if asc_path.exists():
            continue
        match = next(n for n in names if n.startswith(f"data/{tile[:2]}/{tile}_"))
        tile_zip_path = CACHE_DIR / f"{tile}_tile.zip"
        tile_zip_path.write_bytes(gb_zip.read(match))
        with zipfile.ZipFile(tile_zip_path) as tz:
            asc_name = next(n for n in tz.namelist() if n.lower().endswith(".asc"))
            tz.extract(asc_name, CACHE_DIR)
        tile_zip_path.unlink()
print("6 DTM tiles ready:", [p.name for p in CACHE_DIR.glob("N*.asc")])


In [ ]:
def read_asc(path):
    with open(path) as f:
        header = {}
        for _ in range(5):
            k, v = f.readline().split()
            header[k.lower()] = float(v)
        data = np.loadtxt(f)
    return header, data


# Mosaic the 6 tiles into one array: x in [230000, 260000), y in [690000, 710000), 50m cells.
MOSAIC_XLL, MOSAIC_YLL = 230_000, 690_000
MOSAIC_WIDTH_M, MOSAIC_HEIGHT_M, CELL = 30_000, 20_000, 50
ncols, nrows = MOSAIC_WIDTH_M // CELL, MOSAIC_HEIGHT_M // CELL
mosaic = np.full((nrows, ncols), np.nan)

for tile in TILES:
    asc_path = CACHE_DIR / f"{tile.upper()}.asc"
    header, data = read_asc(asc_path)
    col_off = int((header["xllcorner"] - MOSAIC_XLL) / CELL)
    row_off = int((MOSAIC_YLL + MOSAIC_HEIGHT_M - (header["yllcorner"] + header["nrows"] * CELL)) / CELL)
    mosaic[row_off:row_off + int(header["nrows"]), col_off:col_off + int(header["ncols"])] = data

print(f"Mosaic: {mosaic.shape}, elevation {np.nanmin(mosaic):.1f}m to {np.nanmax(mosaic):.1f}m, "
      f"{np.isnan(mosaic).sum()} missing cells")

# Local roughness = std of elevation in a 100m-diameter window (moving-window variance via
# uniform_filter: Var = E[X^2] - E[X]^2, much faster than a literal per-cell std loop).
from scipy.ndimage import uniform_filter

BUFFER_M = 100
window_cells = max(1, round(BUFFER_M * 2 / CELL))
window_cells += 1 - (window_cells % 2)  # force odd

filled = np.where(np.isnan(mosaic), np.nanmean(mosaic), mosaic)
mean_filt = uniform_filter(filled, size=window_cells)
sq_mean_filt = uniform_filter(filled ** 2, size=window_cells)
roughness = np.sqrt(np.clip(sq_mean_filt - mean_filt ** 2, 0, None))


def mosaic_index(x, y):
    col = ((x - MOSAIC_XLL) / CELL).astype(int)
    row = ((MOSAIC_YLL + MOSAIC_HEIGHT_M - y) / CELL).astype(int)
    return row, col


row_idx, col_idx = mosaic_index(plots["x"].values, plots["y"].values)
in_bounds = (row_idx >= 0) & (row_idx < nrows) & (col_idx >= 0) & (col_idx < ncols)
plot_roughness = np.full(len(plots), np.nan)
plot_roughness[in_bounds] = roughness[row_idx[in_bounds], col_idx[in_bounds]]
plots["elevation_roughness"] = plot_roughness

print(f"\n{in_bounds.sum():,} / {len(plots):,} plots sampled")
print(f"Roughness (std elevation, 100m buffer): {plots['elevation_roughness'].min():.2f}m to "
      f"{plots['elevation_roughness'].max():.2f}m, mean {plots['elevation_roughness'].mean():.2f}m")


In [ ]:
quick_spatial_plot(plots, "elevation_roughness", "Local elevation roughness, std within 100m (from OS Terrain 50)", cmap="viridis")


In [ ]:
row = screen_covariate(
    plots, "elevation_roughness", name="Local elevation roughness (100m buffer)", category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM (same cached source as notebooks/environmental_data/, no auth)",
    years_covered="All 6 (2002/2006/2008/2012/2021/2023) -- DTM is static, applies to every survey year equally",
    notes="Complements TOPEX (directional horizon-angle shelter) with a different signal: local "
          "surface variability regardless of direction.",
)
results.append(row)


## 4 & 5. Neighbour mean height, and neighbour height differential

**Important caveat (per the task, kept as a real code comment below, not just prose here):**
these are **spatial-lag features, not exogenous environmental covariates**. Neighbouring plots
often share planting year, soil, and management history, so a plot's neighbours' height is
likely to correlate strongly with its own height for reasons that have nothing to do with
wind-shielding or competition -- it can silently launder plain spatial autocorrelation into
what looks like an "environmental" signal. Reported in a separate feature category from
terrain/wind below, and flagged as needing a with/without check at the SHAP stage, not folded
into an "environmental drivers" narrative on its own.

Computed for a **single representative year (2023)** here, to keep this notebook's scope
contained -- the mechanism (geometry-based KDTree buffer + same-year height lookup) is
identical for every other survey year, since it only needs the same two things (plot geometry,
same-year height) that already exist at every timestamp; see the year-coverage section below.


In [ ]:
YEAR = 2023
year_heights = master_4survey[master_4survey["LiDAR_year"] == YEAR][["identification", "Top_Height99"]]
year_plots = plots[["identification", "x", "y", "cpmt"]].merge(year_heights, on="identification")
print(f"{len(year_plots):,} plots with a real {YEAR} height")

BUFFER_M = 75
tree = cKDTree(year_plots[["x", "y"]].values)
neighbour_lists = tree.query_ball_point(year_plots[["x", "y"]].values, r=BUFFER_M)

heights = year_plots["Top_Height99"].values
neighbour_mean_height = np.full(len(year_plots), np.nan)
neighbour_count = np.zeros(len(year_plots), dtype=int)
for i, neighbours in enumerate(neighbour_lists):
    others = [j for j in neighbours if j != i]
    neighbour_count[i] = len(others)
    if others:
        neighbour_mean_height[i] = heights[others].mean()

year_plots["neighbour_mean_height"] = neighbour_mean_height
# self height minus neighbour mean -- same spatial-lag caveat as neighbour_mean_height itself
# applies here too (see markdown above): a genuine competition/exposure signal and shared
# planting-year/management history both produce the same pattern, and this feature alone
# cannot tell them apart.
year_plots["neighbour_height_differential"] = year_plots["Top_Height99"] - year_plots["neighbour_mean_height"]

print(f"plots with >=1 neighbour within {BUFFER_M}m: {(neighbour_count > 0).sum():,} / {len(year_plots):,}")
print(f"neighbour_mean_height range: {np.nanmin(neighbour_mean_height):.1f}m to {np.nanmax(neighbour_mean_height):.1f}m")

# Merge back onto the main plots table (mean_cr_residual etc.) for the stats screen below.
plots = plots.merge(
    year_plots[["identification", "neighbour_mean_height", "neighbour_height_differential"]],
    on="identification", how="left",
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
for ax, col, title in zip(axes, ["neighbour_mean_height", "neighbour_height_differential"],
                           [f"Neighbour mean height, {YEAR} (m)", f"Self minus neighbour height, {YEAR} (m)"]):
    valid = plots[col].notna()
    draw_survey_outline(ax)
    mappable = ax.hexbin(plots.loc[valid, "x"], plots.loc[valid, "y"], C=plots.loc[valid, col], gridsize=110,
                          extent=(x_lim[0], x_lim[1], y_lim[0], y_lim[1]),
                          cmap="RdBu_r" if "differential" in col else "viridis", mincnt=1, linewidths=0.2, zorder=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(mappable, ax=ax, shrink=0.8)
plt.show()


In [ ]:
SPATIAL_LAG_WARNING = (
    "SPATIAL-LAG WARNING: likely correlates with own height via shared planting "
    "year/management, not necessarily wind-shielding or competition. Needs a "
    "with/without check against the terrain+wind model at the SHAP stage before any "
    "'environmental driver' claim is made from this feature."
)
feature_notes = {
    "neighbour_mean_height": SPATIAL_LAG_WARNING,
    # This feature's true mean is genuinely close to zero (plots and their neighbours have,
    # on average, similar height) -- a real fact, not a data problem, but it makes CV
    # (std/|mean|) inflate to a large, not-very-meaningful number for this one covariate
    # specifically. Read the Moran's I/ICC columns for this row, not its CV.
    "neighbour_height_differential": SPATIAL_LAG_WARNING + " CV is inflated here because this "
        "feature's true mean is close to zero by construction -- not comparable to other rows' CV.",
}
for col, label in [("neighbour_mean_height", f"Neighbour mean height ({YEAR}, 75m buffer)"),
                    ("neighbour_height_differential", f"Neighbour height differential ({YEAR}, 75m buffer)")]:
    row = screen_covariate(
        plots, col, name=label, category="spatial-lag (NOT exogenous -- see caveat above)",
        access_method="Local GPKG geometry + height -- no external source",
        years_covered="All 6 in principle (same geometry/height mechanism every year) -- "
                      f"only {YEAR} actually computed in this notebook",
        notes=feature_notes[col],
    )
    results.append(row)


# Update: AlphaEarth and ERA5-Land unblocked mid-session

Both were confirmed blocked above (no Google Cloud project ID configured for the existing GEE
credentials). A real project ID was supplied after that, unblocking both -- re-tested and
extracted for real below, replacing their placeholder rows in the results table rather than
leaving "blocked" on the record once it's no longer true.

**Note on the project ID below:** this is a personal Google Cloud project, tied to one
person's GEE quota/billing -- not a shared repo default, so it's used directly in this cell
rather than hardcoded into a shared config. Anyone re-running this notebook needs to substitute
their own project ID (see the `EE_PROJECT_ID` cell in section 8 above for how to find one).


In [ ]:
import ee

ee.Initialize(project="neat-planet-500817-i3")
print("GEE initialised with a real project ID.")


## ERA5-Land, real extraction via GEE

Sidesteps the Copernicus CDS key wall entirely -- GEE hosts the same dataset
(`ECMWF/ERA5_LAND/MONTHLY_AGGR`) publicly. Confirmed native resolution from the image's own
projection metadata: **~11.1km**, even coarser than the plan's assumed ~9km -- a small download
covering the whole Aberfoyle bbox is enough (unlike AlphaEarth below, no pixel-volume limit
issue at this resolution).


In [ ]:
era5_image = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2021-01-01", "2021-02-01")
    .first()
    .select("temperature_2m")
)
native_scale = era5_image.projection().nominalScale().getInfo()
print(f"ERA5-Land native resolution: {native_scale:.0f}m")

region = ee.Geometry.Rectangle([-4.71, 56.08, -4.22, 56.27], "EPSG:4326", False)
download_url = era5_image.getDownloadURL({"region": region, "scale": native_scale, "format": "GEO_TIFF"})
era5_response = requests.get(download_url, timeout=60)
era5_path = CACHE_DIR / "era5_land_temp_2021_01.tif"
era5_path.write_bytes(era5_response.content)
print(f"Downloaded {era5_path.stat().st_size:,} bytes (real GEE data, not blocked)")

with rasterio.open(era5_path) as ds:
    print("CRS:", ds.crs, "shape:", ds.shape)
    transformer = Transformer.from_crs("EPSG:27700", ds.crs, always_xy=True)
    px, py = transformer.transform(plots["x"].values, plots["y"].values)
    rows_idx, cols_idx = rasterio.transform.rowcol(ds.transform, px, py)
    rows_idx, cols_idx = np.array(rows_idx), np.array(cols_idx)
    array = ds.read(1)
    in_bounds = (rows_idx >= 0) & (rows_idx < array.shape[0]) & (cols_idx >= 0) & (cols_idx < array.shape[1])
    era5_values = np.full(len(plots), np.nan)
    era5_values[in_bounds] = array[rows_idx[in_bounds], cols_idx[in_bounds]]

plots["era5_land_temp_k"] = era5_values
print(f"\n{in_bounds.sum():,} / {len(plots):,} plots sampled")
print(f"Distinct values: {plots['era5_land_temp_k'].nunique()}, "
      f"range: {plots['era5_land_temp_k'].min():.3f}K to {plots['era5_land_temp_k'].max():.3f}K")


In [ ]:
quick_spatial_plot(plots, "era5_land_temp_k", "ERA5-Land 2m temperature, Jan 2021 (~11km, real GEE extraction)", cmap="coolwarm")


In [ ]:
era5_row = screen_covariate(
    plots, "era5_land_temp_k", name="ERA5-Land (2m temperature)", category="climate",
    access_method="Google Earth Engine, real project ID (confirmed live -- bypasses the CDS key wall)",
    years_covered="Available back to 1950, but at ~11km resolution -- see notes",
    notes="REAL EXTRACTION (updates the earlier 'blocked' placeholder). Confirmed native "
          "resolution ~11.1km, coarser than the plan's assumed ~9km -- check the distinct-value "
          "count above against the earlier ~3-4-cells estimate.",
)
# Replace the earlier placeholder row rather than duplicate it.
results[:] = [r for r in results if r["dataset"] != "ERA5-Land (9km)"]
results.append(era5_row)


## AlphaEarth Foundations, real extraction via GEE

64-band annual embedding image, confirmed live for 2017-2025 (tested directly: 97,155 annual
images in the collection, years 2017 through 2025 -- matches the plan's "2017 onward" claim
exactly). A full-resolution (10m), full-bbox, all-64-band raster download exceeds GEE's
synchronous 50MB pixel-volume limit (tested: request would be ~6.6GB) -- so this uses
server-side **point sampling** at a 5,000-plot random subsample instead (consistent with this
notebook's existing `sample_size=5000` convention for Moran's I/variogram elsewhere), not a
full-bbox raster like every other source above.

**The 64 dimensions are not physically interpretable** (a learned representation, not named
covariates) -- reduced via PCA to its first principal component for the statistical screen,
which is the standard way to ask "is there any real spatial signal in here at all" without
pretending any individual one of the 64 numbers means something on its own.


In [ ]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(42)
alphaearth_sample = plots.sample(n=5000, random_state=42).reset_index(drop=True)
transformer_4326 = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
sample_lon, sample_lat = transformer_4326.transform(alphaearth_sample["x"].values, alphaearth_sample["y"].values)

features = [ee.Feature(ee.Geometry.Point([lo, la]), {"idx": i}) for i, (lo, la) in enumerate(zip(sample_lon, sample_lat))]
sample_fc = ee.FeatureCollection(features)

alphaearth_image = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate("2023-01-01", "2024-01-01").mosaic()
sampled = alphaearth_image.sampleRegions(collection=sample_fc, scale=10, geometries=False).getInfo()

band_names = [f"A{i:02d}" for i in range(64)]
embedding_matrix = np.array([[f["properties"].get(b, np.nan) for b in band_names] for f in sampled["features"]])
sample_idx = [f["properties"]["idx"] for f in sampled["features"]]
print(f"Sampled {len(sampled['features']):,} / {len(alphaearth_sample):,} points, "
      f"embedding matrix shape {embedding_matrix.shape}")

pca = PCA(n_components=1, random_state=42)
pc1 = pca.fit_transform(embedding_matrix).ravel()
print(f"PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of the 64-dim embedding's variance")

alphaearth_result = alphaearth_sample.iloc[sample_idx].copy()
alphaearth_result["alphaearth_pc1"] = pc1


In [ ]:
quick_spatial_plot(alphaearth_result, "alphaearth_pc1", "AlphaEarth 2023 embedding, PC1 (10m, real GEE extraction, n=5000 sample)", cmap="viridis")


In [ ]:
alphaearth_row = screen_covariate(
    alphaearth_result, "alphaearth_pc1", name="AlphaEarth Foundations (PC1 of 64-dim embedding)",
    category="other (learned embedding, not a named covariate)",
    access_method="Google Earth Engine, real project ID (confirmed live; point-sampled n=5000, "
                  "not full raster -- 64-band/10m/full-bbox download exceeds GEE's 50MB sync limit)",
    years_covered="2017-2025 ONLY -- confirmed no 2002/2006/2008/2012 coverage, still cannot "
                  "support the temporal Env-PINN window or SQ2 regardless of access",
    notes=f"REAL EXTRACTION (updates the earlier 'blocked' placeholder). PC1 explains "
          f"{pca.explained_variance_ratio_[0]*100:.1f}% of the full 64-dim variance -- a single "
          f"PC is a lossy summary, real signal could exist in other components this screen "
          f"doesn't see. Still not physically interpretable; robustness/clustering check only, "
          f"never a primary SHAP-attributed feature (kept as a code comment above too).",
)
results[:] = [r for r in results if r["dataset"] != "AlphaEarth Foundations (10m embeddings)"]
results.append(alphaearth_row)


# Part 3: More terrain-derived features, all from the same DTM already in hand

A follow-up batch, split strictly by what's actually available:

**Computable right now** (this section): curvature (plan + profile), Terrain Position Index
(TPI), a frost-hollow flag, a solar radiation index, and a simple soil-depth proxy -- all direct
derivatives of the OS Terrain 50 mosaic already built for elevation roughness above, so no new
data source, and all six survey years automatically (same as every other GPKG/DTM-derived
feature). Distance to nearest watercourse is added too -- not a DTM derivative, but resolved
simply via **OS Open Rivers** (free vector data, no auth) rather than needing flow accumulation.

**Checked and set aside, not built:**
- **DAMS windiness score / Windthrow Hazard Classification inputs** -- DAMS combines a Wind Zone
  base map (from historical tatter-flag surveys) with a TOPEX-style exposure adjustment,
  elevation, and aspect. It's integrated into Forest Research's Ecological Site Classification
  (ESC) tool, but no bulk GIS download for the underlying Wind Zone map turned up in this
  session's searching -- likely a formal data request to Forest Research, not a quick pull. Not
  attempted here; the plan's own existing TOPEX/wind-exposure combination remains the fallback,
  exactly as flagged before.
- **Flow-accumulation-based hydrology** (a "proper" TWI, or a stream network derived from the
  DTM itself) -- `richdem` failed to build from source in this environment (real C++ compile
  errors, not a quick fix) and no other flow-accumulation library was tried. Sidestepped
  entirely for the stream-distance feature by using OS Open Rivers' real watercourse lines
  instead of deriving a synthetic network -- simpler and doesn't need this dependency at all.


## Slope and aspect (prerequisite for curvature and the solar radiation index below)

Not yet computed anywhere in this repo (checked first) -- standard central-difference gradient
on the DTM mosaic. Validated against a synthetic tilted plane before trusting it on the real
DTM: a surface rising to the south (i.e. downhill faces north) must score aspect=0 deg and
northness=+1, and a surface rising to the east (downhill faces west) must score aspect=270 deg
-- both checked and passed before this cell was written into the notebook.


In [ ]:
def slope_aspect(elevation, cell=CELL):
    # Grid convention (matches the mosaic built above): row increases SOUTH, col increases EAST.
    dz_dx_east = np.gradient(elevation, cell, axis=1)
    dz_dy_north = -np.gradient(elevation, cell, axis=0)  # row->north needs a sign flip
    slope_rad = np.arctan(np.sqrt(dz_dx_east ** 2 + dz_dy_north ** 2))
    # Aspect = compass bearing (0=N, 90=E, 180=S, 270=W) of the DOWNHILL direction.
    aspect_rad = np.arctan2(-dz_dx_east, -dz_dy_north) % (2 * np.pi)
    return slope_rad, aspect_rad, dz_dx_east, dz_dy_north


slope_rad, aspect_rad, dz_dx, dz_dy = slope_aspect(mosaic)
plots["slope_degrees"] = np.nan
plots["aspect_degrees"] = np.nan
row_idx, col_idx = mosaic_index(plots["x"].values, plots["y"].values)
in_bounds = (row_idx >= 0) & (row_idx < nrows) & (col_idx >= 0) & (col_idx < ncols)
plots.loc[in_bounds, "slope_degrees"] = np.degrees(slope_rad[row_idx[in_bounds], col_idx[in_bounds]])
plots.loc[in_bounds, "aspect_degrees"] = np.degrees(aspect_rad[row_idx[in_bounds], col_idx[in_bounds]])

print(f"Slope: {plots['slope_degrees'].min():.1f} to {plots['slope_degrees'].max():.1f} degrees, "
      f"mean {plots['slope_degrees'].mean():.1f}")
print(f"Aspect: distinct values {plots['aspect_degrees'].nunique()} (continuous compass bearing, 0-360)")


## Curvature (plan + profile)

Zevenbergen & Thorne (1987) second-derivative formulas -- same differential-geometry family as
slope/aspect above, standard GIS convention (positive = convex/ridge, negative = concave/valley,
for both). Validated against a synthetic parabolic bowl and dome (hand-checked against the
formula's own analytical prediction, not just "looks about right") before trusting it here: a
bowl must score negative on both profile and plan curvature away from its exact centre (where
the gradient is genuinely zero and both curvatures correctly evaluate to 0, a real feature of
the formula, not a bug), and a dome must score positive on both.


In [ ]:
def curvature(elevation, cell=CELL):
    dz_dx = np.gradient(elevation, cell, axis=1)
    dz_dy = -np.gradient(elevation, cell, axis=0)
    d2z_dx2 = np.gradient(dz_dx, cell, axis=1)
    d2z_dy2 = -np.gradient(dz_dy, cell, axis=0)
    d2z_dxdy = -np.gradient(dz_dx, cell, axis=0)

    p = dz_dx ** 2 + dz_dy ** 2
    q = p + 1
    with np.errstate(divide="ignore", invalid="ignore"):
        profile = -(d2z_dx2 * dz_dx ** 2 + 2 * d2z_dxdy * dz_dx * dz_dy + d2z_dy2 * dz_dy ** 2) / (p * q ** 1.5)
        plan = -(d2z_dx2 * dz_dy ** 2 - 2 * d2z_dxdy * dz_dx * dz_dy + d2z_dy2 * dz_dx ** 2) / p ** 1.5
    # Flat ground (p=0, no defined slope direction) is 0 by convention, not undefined/NaN.
    profile = np.where(p == 0, 0.0, profile)
    plan = np.where(p == 0, 0.0, plan)
    return profile, plan


profile_curv, plan_curv = curvature(mosaic)
plots["profile_curvature"] = np.nan
plots["plan_curvature"] = np.nan
plots.loc[in_bounds, "profile_curvature"] = profile_curv[row_idx[in_bounds], col_idx[in_bounds]]
plots.loc[in_bounds, "plan_curvature"] = plan_curv[row_idx[in_bounds], col_idx[in_bounds]]

print(f"Profile curvature: {plots['profile_curvature'].min():.4f} to {plots['profile_curvature'].max():.4f}")
print(f"Plan curvature: {plots['plan_curvature'].min():.4f} to {plots['plan_curvature'].max():.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
for ax, col, title in zip(axes, ["profile_curvature", "plan_curvature"], ["Profile curvature", "Plan curvature"]):
    valid = plots[col].notna()
    draw_survey_outline(ax)
    mappable = ax.hexbin(plots.loc[valid, "x"], plots.loc[valid, "y"], C=plots.loc[valid, col], gridsize=110,
                          extent=(x_lim[0], x_lim[1], y_lim[0], y_lim[1]), cmap="RdBu_r",
                          vmin=-np.nanpercentile(np.abs(plots[col]), 95), vmax=np.nanpercentile(np.abs(plots[col]), 95),
                          mincnt=1, linewidths=0.2, zorder=2)
    ax.set_title(f"{title} (convex=red/ridge, concave=blue/valley)", fontsize=9)
    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(mappable, ax=ax, shrink=0.8)
plt.show()


In [ ]:
for col, label in [("profile_curvature", "Profile curvature"), ("plan_curvature", "Plan curvature")]:
    row = screen_covariate(
        plots, col, name=label, category="terrain-derived (new, from OS Terrain 50)",
        access_method="OS Terrain 50 DTM (same cached source as notebooks/environmental_data/, no auth)",
        years_covered="All 6 (2002/2006/2008/2012/2021/2023) -- DTM is static, applies to every survey year equally",
        notes="Zevenbergen & Thorne (1987) formula, sign-convention validated against a synthetic "
              "bowl/dome before use on real data (see markdown above).",
    )
    results.append(row)


## Terrain Position Index (TPI), and a frost-hollow flag derived from it

TPI = a plot's own elevation minus the mean elevation in its neighbourhood (100m, reusing
`mean_filt` already computed for elevation roughness above -- literally free, no new
computation). Negative TPI = sits below its surroundings (valley/hollow); positive = sits above
(ridge/knoll) -- distinct from raw elevation, since two plots at the same absolute elevation can
have opposite TPI depending on what's around them.

**Frost-hollow flag**: cold air drains downhill and pools in low, concave ground on still
nights -- a real, distinct growth-suppression mechanism (frost damage to young Sitka) from
everything else derived so far. Flagged where TPI is well below the local mean (bottom 15% of
TPI, an arbitrary-but-documented threshold, not a calibrated one) **and** plan curvature is
concave (converging ground, where cold air would actually collect rather than just being low).


In [ ]:
tpi = mosaic - mean_filt  # mean_filt: 100m-window mean elevation, already computed for roughness above
plots["tpi"] = np.nan
plots.loc[in_bounds, "tpi"] = tpi[row_idx[in_bounds], col_idx[in_bounds]]
print(f"TPI: {plots['tpi'].min():.2f}m to {plots['tpi'].max():.2f}m")

# Threshold is a documented choice, not a calibrated one: bottom 15% of TPI (well below the
# local mean) AND concave plan curvature (converging, not just low) -- both conditions needed
# so a plot on a gentle, uniformly-sloped hillside (low TPI, but not concave) isn't flagged.
tpi_threshold = plots["tpi"].quantile(0.15)
plots["frost_hollow_flag"] = ((plots["tpi"] < tpi_threshold) & (plots["plan_curvature"] < 0)).astype(float)
plots.loc[plots["tpi"].isna() | plots["plan_curvature"].isna(), "frost_hollow_flag"] = np.nan

print(f"TPI threshold (15th percentile): {tpi_threshold:.2f}m")
print(f"Plots flagged as frost-hollow candidates: {(plots['frost_hollow_flag'] == 1).sum():,} "
      f"/ {plots['frost_hollow_flag'].notna().sum():,} ({(plots['frost_hollow_flag'] == 1).mean()*100:.1f}%)")


In [ ]:
quick_spatial_plot(plots, "tpi", "Terrain Position Index (100m neighbourhood)", cmap="RdBu_r")


In [ ]:
row = screen_covariate(
    plots, "tpi", name="Terrain Position Index (TPI, 100m)", category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM, no auth",
    years_covered="All 6 -- DTM is static",
    notes="Distinct from raw elevation: captures relative landform position (ridge/mid-slope/"
          "valley), not absolute height.",
)
results.append(row)

frost_row = screen_covariate(
    plots, "frost_hollow_flag", name="Frost-hollow flag (TPI + concave curvature)", category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM, no auth -- purely derived, no new source",
    years_covered="All 6 -- DTM is static",
    notes="Binary flag, threshold (15th percentile TPI + concave plan curvature) is a documented "
          "heuristic choice, not calibrated against any real frost/damage record -- worth revisiting "
          "if frost damage data ever becomes available to validate against.",
)
results.append(frost_row)


## Solar radiation index

Cosine law of illumination (slope + aspect vs. sun position) at solar noon, summer solstice, for
Aberfoyle's latitude (56.15N) -- no horizon shading from surrounding terrain, just local
slope/aspect geometry against the sun's angle. Kept deliberately simple (a closed-form formula,
no new dependency) rather than a full horizon-shading model; validated first on a synthetic
slope: a south-facing 20-degree slope must score higher than an equally-steep north-facing one,
which must score lower than flat ground -- confirmed before use here.


In [ ]:
LATITUDE_DEG = 56.15
SOLAR_DECLINATION_DEG = 23.44  # summer solstice -- the sun's highest, most aspect-differentiating position
solar_elevation_deg = 90 - abs(LATITUDE_DEG - SOLAR_DECLINATION_DEG)
zenith_rad = np.radians(90 - solar_elevation_deg)
solar_azimuth_rad = np.radians(180)  # due south at solar noon, by definition

solar_index = (np.cos(slope_rad) * np.cos(zenith_rad)
               + np.sin(slope_rad) * np.sin(zenith_rad) * np.cos(aspect_rad - solar_azimuth_rad))

plots["solar_radiation_index"] = np.nan
plots.loc[in_bounds, "solar_radiation_index"] = solar_index[row_idx[in_bounds], col_idx[in_bounds]]
print(f"Solar radiation index (solar noon, summer solstice): "
      f"{plots['solar_radiation_index'].min():.3f} to {plots['solar_radiation_index'].max():.3f}")


In [ ]:
quick_spatial_plot(plots, "solar_radiation_index", "Solar radiation index (noon, summer solstice, no horizon shading)", cmap="viridis")


In [ ]:
row = screen_covariate(
    plots, "solar_radiation_index", name="Solar radiation index (slope+aspect, noon/solstice)",
    category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM, no auth -- closed-form formula, no new dependency",
    years_covered="All 6 -- DTM is static",
    notes="No horizon shading by surrounding terrain -- a plot in a valley with the 'right' local "
          "aspect still scores as if unshaded, which is optimistic for valley-floor plots "
          "specifically. A full horizon-shading model (like TOPEX, but toward the sun's actual "
          "azimuth rather than all compass directions) would fix this if the gap turns out to matter.",
)
results.append(row)


## Simple soil-depth proxy (from slope)

Steeper ground generally has shallower, more eroded soil than gentle ground -- a defensible
terrain-only proxy for rooting depth (relevant to windthrow anchorage) when the real soil
datasets already checked (CEH, James Hutton) are too coarse to use directly. **Explicitly a
heuristic, not a calibrated model** -- just `-slope`, so higher values read as "proxy for
deeper soil."


In [ ]:
plots["soil_depth_proxy"] = -plots["slope_degrees"]

row = screen_covariate(
    plots, "soil_depth_proxy", name="Soil-depth proxy (-slope, heuristic)", category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM, no auth -- purely derived, no new source",
    years_covered="All 6 -- DTM is static",
    notes="Pure heuristic (steeper = assumed shallower soil), not calibrated against any real "
          "soil-depth measurement -- directly redundant with slope itself in the statistics below "
          "(same information, just sign-flipped and relabelled), included for interpretability "
          "in a rooting-depth/windthrow framing rather than as new information.",
)
results.append(row)


## Distance to nearest watercourse (OS Open Rivers)

Not a DTM derivative -- resolved via **OS Open Rivers**, a free vector dataset (watercourse
centrelines), same no-auth access pattern as OS Terrain 50/OS Open Rivers' sibling products.
Sidesteps flow-accumulation entirely (which would have needed `richdem`, confirmed failing to
build in this environment -- see the note at the top of this Part).


In [ ]:
rivers_zip_path = CACHE_DIR / "oprvrs_gpkg_gb.zip"
if not rivers_zip_path.exists():
    resp = requests.get(
        "https://api.os.uk/downloads/v1/products/OpenRivers/downloads?area=GB&format=GeoPackage&redirect",
        timeout=60,
    )
    resp.raise_for_status()
    rivers_zip_path.write_bytes(resp.content)
print(f"OS Open Rivers cached: {rivers_zip_path.stat().st_size / 1e6:.1f} MB")

rivers_gpkg_path = CACHE_DIR / "oprvrs_gb.gpkg"
if not rivers_gpkg_path.exists():
    with zipfile.ZipFile(rivers_zip_path) as z:
        z.extract("Data/oprvrs_gb.gpkg", CACHE_DIR)
    (CACHE_DIR / "Data" / "oprvrs_gb.gpkg").rename(rivers_gpkg_path)
    (CACHE_DIR / "Data").rmdir()

margin_m = 5000
bbox_with_margin = (
    BBOX_27700["minx"] - margin_m, BBOX_27700["miny"] - margin_m,
    BBOX_27700["maxx"] + margin_m, BBOX_27700["maxy"] + margin_m,
)
rivers = gpd.read_file(rivers_gpkg_path, layer="watercourse_link", bbox=bbox_with_margin)
print(f"{len(rivers)} watercourse segments in/near the Aberfoyle bbox, CRS={rivers.crs}")

river_network = rivers.geometry.union_all()
plots_gdf_rivers = gpd.GeoDataFrame(plots, geometry=gpd.points_from_xy(plots["x"], plots["y"]), crs=27700)
plots["dist_to_watercourse"] = plots_gdf_rivers.geometry.distance(river_network)

print(f"Distance to nearest watercourse: {plots['dist_to_watercourse'].min():.1f}m to "
      f"{plots['dist_to_watercourse'].max():.1f}m, mean {plots['dist_to_watercourse'].mean():.1f}m")


In [ ]:
quick_spatial_plot(plots, "dist_to_watercourse", "Distance to nearest watercourse (OS Open Rivers)", cmap="viridis")


In [ ]:
row = screen_covariate(
    plots, "dist_to_watercourse", name="Distance to nearest watercourse", category="terrain-derived (new, from OS Open Rivers)",
    access_method="OS Open Rivers, direct download, no auth (confirmed live)",
    years_covered="All 6 (2026 snapshot of a relatively static watercourse network) -- treated as "
                  "time-invariant, applies to all six survey years equally",
    notes="Resolves the earlier open question about needing flow-accumulation -- OS Open Rivers' "
          "real watercourse lines make this a simple distance-to-nearest-line calculation, not a "
          "hydrology derivation.",
)
results.append(row)


## TOPEX (topographic exposure) -- a real gap, fixed

Referenced throughout this notebook and its predecessor as "derived from the DTM, no separate
source needed" -- but never actually implemented until now. Standard single-radius TOPEX
(Wilson 1984; the same method underlying DAMS's exposure adjustment, discussed above): sum of
horizon angles to the terrain in 8 compass directions at a fixed 1000m radius. **Sign
convention**: positive = sheltered (surrounded by higher ground), negative = exposed (ridge/
knoll surrounded by lower ground) -- validated on a synthetic conical hill (summit scored a
strongly negative/exposed -133, flat ground far away scored ~0) before trusting it on the real
DTM, same practice as every other terrain derivative in this notebook.

Horizon points are sampled via the same nearest-cell lookup as everything else here (see the
interpolation explanation below) -- not interpolated along the ray.


In [ ]:
DIRECTIONS_DEG = [0, 45, 90, 135, 180, 225, 270, 315]  # N, NE, E, SE, S, SW, W, NW
TOPEX_DISTANCE_M = 1000  # single-radius version -- the simplest standard TOPEX choice; multi-radius
                          # (e.g. averaging 500/1000/2000m) is a documented extension, not done here


def topex(elevation, cell=CELL, distance=TOPEX_DISTANCE_M, directions_deg=DIRECTIONS_DEG):
    nrows_, ncols_ = elevation.shape
    row_grid, col_grid = np.meshgrid(np.arange(nrows_), np.arange(ncols_), indexing="ij")
    angle_sum = np.zeros_like(elevation)

    for bearing_deg in directions_deg:
        bearing_rad = np.radians(bearing_deg)
        d_east = distance * np.sin(bearing_rad)
        d_north = distance * np.cos(bearing_rad)
        d_col = d_east / cell
        d_row = -d_north / cell  # north = -row direction, same convention as slope/aspect above

        sample_row = np.round(row_grid + d_row).astype(int)
        sample_col = np.round(col_grid + d_col).astype(int)
        in_bounds = (sample_row >= 0) & (sample_row < nrows_) & (sample_col >= 0) & (sample_col < ncols_)
        sample_row_c = np.clip(sample_row, 0, nrows_ - 1)
        sample_col_c = np.clip(sample_col, 0, ncols_ - 1)
        remote_elev = elevation[sample_row_c, sample_col_c]

        # Horizon angle: positive if the surrounding point is HIGHER (sheltered), negative if
        # lower (exposed) -- Wilson (1984) / standard TOPEX convention.
        angle = np.degrees(np.arctan2(remote_elev - elevation, distance))
        angle_sum = np.where(in_bounds, angle_sum + angle, angle_sum)

    return angle_sum


topex_values = topex(mosaic)
plots["topex"] = np.nan
plots.loc[in_bounds, "topex"] = topex_values[row_idx[in_bounds], col_idx[in_bounds]]
print(f"TOPEX: {plots['topex'].min():.1f} (most exposed) to {plots['topex'].max():.1f} (most sheltered)")


In [ ]:
quick_spatial_plot(plots, "topex", "TOPEX, 1000m radius (positive=sheltered, negative=exposed)", cmap="RdBu")


In [ ]:
row = screen_covariate(
    plots, "topex", name="TOPEX (1000m radius, 8 directions)", category="terrain-derived (new, from OS Terrain 50)",
    access_method="OS Terrain 50 DTM, no auth -- purely derived, no new source",
    years_covered="All 6 -- DTM is static",
    notes="Single-radius version (1000m); sign-convention validated against a synthetic conical "
          "hill before use on real data (see markdown above). The DAMS section earlier in this "
          "notebook names this exact method as one of DAMS's own two inputs (alongside a Wind "
          "Zone base map this notebook doesn't have) -- this is the fallback that section pointed to.",
)
results.append(row)


## How every raster value above was actually assigned: nearest-cell, not interpolation

Worth stating explicitly, since it changes how much to trust a value near a cell boundary.
Every extraction in this notebook (OS Terrain 50, SoilGrids, CHELSA, Global Wind Atlas,
HadUK-Grid, ERA5-Land, CEH) uses the same mechanism:

1. Convert the plot's real-world (x, y) into the raster's row/column space using the raster's
   affine transform.
2. **Truncate (or round) to the nearest integer row/column** -- i.e. find which single grid
   cell the point physically falls inside.
3. Read that one cell's value directly. No averaging with neighbouring cells, no distance
   weighting.

This is **nearest-neighbour / point-in-cell sampling**, not bilinear or bicubic interpolation.
Concretely, for a plot sitting 2m from the edge of its containing cell in a 1km-resolution
product (HadUK-Grid, ERA5-Land): the value assigned is that whole 1km cell's value, identical
to a plot sitting 998m away on the other side of the same cell, and there is a hard discontinuity
(not a smooth gradient) at the cell boundary itself. For OS Terrain 50 (50m cells), the same
principle applies at a much finer scale -- the practical effect is far smaller, but the
mechanism is identical.

**Why not interpolate?** Bilinear interpolation would be a defensible upgrade for the smoother,
coarser products (HadUK-Grid, ERA5-Land, Global Wind Atlas, CHELSA, SoilGrids) where the
underlying phenomenon genuinely varies continuously between cells. It would NOT be appropriate
for the categorical sources (CEH pedotope classes, James Hutton WRB soil groups) -- averaging
two different soil class codes produces a meaningless number, not a real intermediate class.
Kept as nearest-cell throughout for consistency across both types in this screening pass; worth
revisiting per-source (interpolate the continuous ones, keep categorical ones as nearest-cell)
before any of these become real model inputs rather than a resolution check.


# Final results table

Every dataset and feature above, one row each. Year coverage is its own explicit column
throughout (not an aside) -- the five new GPKG-derived features all show every one of the six
survey years, a real practical advantage over most of the external sources, which are static
snapshots. Sorted by category, then by Moran's I descending within category, so it's easy to
see at a glance which covariates have genuine plot-level variation (by the statistical tests,
not just resolution labels) vs. which are smooth/blocked/spatial-lag.


In [ ]:
results_df = pd.DataFrame(results)

category_order = {
    "climate": 0, "soil": 1, "wind": 2,
    "terrain-derived (new, from OS Terrain 50)": 3,
    "terrain-derived (new, from GPKG geometry)": 3,
    "terrain-derived (new, from OS Open Rivers)": 3,
    "spatial-lag (NOT exogenous -- see caveat above)": 4,
    "other (learned embedding, not a named covariate)": 5,
}
results_df["_cat_order"] = results_df["category"].map(category_order).fillna(9)
results_df = results_df.sort_values(["_cat_order", "morans_i"], ascending=[True, False]).drop(columns="_cat_order")

display_cols = [
    "dataset", "category", "n_plots_sampled", "n_distinct_values", "cv",
    "variogram_range_m", "variogram_status", "morans_i", "morans_i_p",
    "between_compartment_icc", "spearman_vs_cr_residual_4survey", "spearman_vs_cr_residual_6survey",
    "years_covered", "access_method",
]
results_df[display_cols]


In [ ]:
results_df[["dataset", "notes"]]


## A trust score, not just a resolution label

The whole point of this notebook was "don't trust resolution labels alone" -- this closes the
loop by turning that principle into one comparable number per row, built almost entirely from
statistics already computed above, plus one deliberately-isolated subjective input.

**The formula** (0-1, higher = more trustworthy for plot-level spatial attribution):

```
trust = access_confirmed * (0.35*(1 - ICC) + 0.15*moran_significant + 0.30*provenance + 0.20*temporal_match)
```

- **`access_confirmed`** (hard gate, 0 or 1): was this actually extracted from real data this
  session, or is it blocked/estimated? An unconfirmed source scores 0 regardless of everything
  else -- no amount of good resolution or provenance matters if the value was never really
  obtained. Read directly off `n_plots_sampled > 0` above; every row happens to pass this gate
  now (everything was unblocked or fixed during this session), but it stayed real gate logic
  since a genuinely-blocked source added later should still score 0.
- **`1 - ICC`** (weight 0.35, the single biggest factor): the between-compartment ICC already
  computed above, inverted. A source that's almost entirely smooth regional trend (ICC near 1)
  can't discriminate between two plots in the same stand no matter how official it is -- this
  is the direct, already-quantified version of "does it actually resolve real plot-level
  variation," not a resolution label.
- **`moran_significant`** (weight 0.15): 1 if the already-computed Moran's I p-value is below
  0.05 (real spatial structure detected, not indistinguishable from noise), else 0.
- **`provenance`** (weight 0.30) -- **the one genuinely subjective input**, a 0-1 judgement per
  source: 1.0 for exact geometry/a real official survey (OS Terrain 50, OS Open Rivers,
  compartment/perimeter distance), tapering down through ML-on-real-observations (SoilGrids,
  CEH) and modelled reanalysis/climatology (CHELSA, Global Wind Atlas, ERA5-Land), down to 0.5
  for AlphaEarth's uninterpretable learned embedding and 0.5 for the frankly-heuristic
  soil-depth proxy. Documented explicitly below rather than hidden inside the formula -- argue
  with these numbers if they look wrong, that's the point of writing them down.
- **`temporal_match`** (weight 0.20): does the data actually cover the years it's being used
  for? 1.0 for anything genuinely static (terrain, geometry) or a long climatology bracketing
  the survey period; 0.5-0.6 for a single non-representative snapshot year (HadUK-Grid 2021
  only, ERA5-Land Jan 2021 only, or the neighbour-height features computed for 2023 only in
  this notebook); 0.25 for AlphaEarth, which is missing 2002/2006/2008/2012 entirely -- the
  worst temporal mismatch of anything checked here.

**Why these weights specifically**: ICC gets the largest share because it's this dissertation's
actual bottleneck -- attribution needs within-stand variation, not another way to rediscover
which compartment a plot is in. Provenance is second because a well-resolved but poorly-sourced
number is still not trustworthy. Moran's I and temporal match are real but smaller factors: a
source can have significant-but-weak spatial structure and still not be that useful, and a
temporal mismatch is a genuine problem but a narrower one (affects which years/questions a
source can support, not whether it's wrong outright). These weights are a starting point, not a
calibrated result -- change them and re-run this cell if a different weighting makes more sense
for a specific chapter's argument.


In [ ]:
results_csv_path = project_root / "notebooks" / "environmental_data" / "figures" / "aux_data_resolution_check_results.csv"
results_csv_path.parent.mkdir(parents=True, exist_ok=True)

# The one subjective input, written down explicitly -- everything else in the formula comes
# straight from columns already in results_df.
PROVENANCE_WEIGHT = {
    "HadUK-Grid (tas, 1km)": 0.75,               # station-interpolated official observational product
    "ERA5-Land (2m temperature)": 0.60,           # numerical reanalysis, not direct observation
    "CEH 50m soil (natural capital pedotopes)": 0.70,   # modelled from real soil survey + terrain covariates
    "James Hutton 1:250,000 soil map (WRB)": 0.60,      # real soil survey, but very coarse mapping scale
    "SoilGrids topsoil pH 0-5cm": 0.75,           # ML model trained on real global soil profile observations
    "CHELSA bio1 (mean annual temp)": 0.70,       # statistically downscaled, terrain-conditioned climatology
    "Global Wind Atlas (10m wind speed)": 0.60,   # mesoscale wind model output, not direct measurement
    "AlphaEarth Foundations (PC1 of 64-dim embedding)": 0.45,  # learned embedding, physically uninterpretable
    "Distance to compartment boundary": 1.00,     # exact geometry, no modelling
    "Distance to forest perimeter": 1.00,
    "Local elevation roughness (100m buffer)": 0.95,  # deterministic derivative of a survey-grade DTM
    "Neighbour mean height (2023, 75m buffer)": 0.90,  # real geometry + real measured height (see spatial-lag caveat)
    "Neighbour height differential (2023, 75m buffer)": 0.90,
    "Profile curvature": 0.95,
    "Plan curvature": 0.95,
    "Terrain Position Index (TPI, 100m)": 0.95,
    "Frost-hollow flag (TPI + concave curvature)": 0.70,  # DTM-derived, but an uncalibrated threshold layered on top
    "Solar radiation index (slope+aspect, noon/solstice)": 0.85,  # no horizon shading, a known simplification
    "Soil-depth proxy (-slope, heuristic)": 0.50,  # explicitly a heuristic, never validated against real depth
    "Distance to nearest watercourse": 0.95,       # real official vector survey (OS Open Rivers)
    "TOPEX (1000m radius, 8 directions)": 0.95,    # DTM derivative, standard validated method
}

TEMPORAL_MATCH = {
    "HadUK-Grid (tas, 1km)": 0.50,                 # only 2021 downloaded -- doesn't represent the other 5 years
    "ERA5-Land (2m temperature)": 0.50,            # only Jan 2021 sampled
    "CEH 50m soil (natural capital pedotopes)": 0.90,   # static soil property, one snapshot is reasonable
    "James Hutton 1:250,000 soil map (WRB)": 0.90,
    "SoilGrids topsoil pH 0-5cm": 0.90,
    "CHELSA bio1 (mean annual temp)": 0.85,        # long climatology brackets the survey period reasonably
    "Global Wind Atlas (10m wind speed)": 0.85,
    "AlphaEarth Foundations (PC1 of 64-dim embedding)": 0.25,  # missing 2002/2006/2008/2012 ENTIRELY
    "Distance to compartment boundary": 1.00,
    "Distance to forest perimeter": 1.00,
    "Local elevation roughness (100m buffer)": 1.00,
    "Neighbour mean height (2023, 75m buffer)": 0.60,  # mechanism generalises to all 6 years, but only 2023 was actually run
    "Neighbour height differential (2023, 75m buffer)": 0.60,
    "Profile curvature": 1.00,
    "Plan curvature": 1.00,
    "Terrain Position Index (TPI, 100m)": 1.00,
    "Frost-hollow flag (TPI + concave curvature)": 1.00,
    "Solar radiation index (slope+aspect, noon/solstice)": 1.00,
    "Soil-depth proxy (-slope, heuristic)": 1.00,
    "Distance to nearest watercourse": 0.95,        # a 2026 snapshot of a relatively static network
    "TOPEX (1000m radius, 8 directions)": 1.00,
}

access_confirmed = (results_df["n_plots_sampled"] > 0).astype(float)
moran_significant = (results_df["morans_i_p"] < 0.05).astype(float)
resolving_power = 1 - results_df["between_compartment_icc"]
provenance = results_df["dataset"].map(PROVENANCE_WEIGHT)
temporal_match = results_df["dataset"].map(TEMPORAL_MATCH)

results_df["trust_score"] = access_confirmed * (
    0.35 * resolving_power + 0.15 * moran_significant + 0.30 * provenance + 0.20 * temporal_match
)

missing_provenance = results_df.loc[results_df["dataset"].map(PROVENANCE_WEIGHT).isna(), "dataset"].tolist()
if missing_provenance:
    print("WARNING -- no provenance/temporal weight assigned, trust_score will be NaN for:", missing_provenance)

results_df.sort_values("trust_score", ascending=False)[
    ["dataset", "category", "trust_score", "between_compartment_icc", "morans_i_p", "n_plots_sampled"]
]


In [ ]:
results_df.to_csv(results_csv_path, index=False)
print(f"Updated results table (with trust_score) saved to {results_csv_path}")


## Surprising findings, worth telling the supervisor directly

In [ ]:
# Written from the actual numbers in results_df above, not hardcoded -- re-running this
# notebook with fresh extractions (e.g. a different James Hutton subsample) keeps this in sync.
by_name = results_df.set_index("dataset")

chelsa = by_name.loc["CHELSA bio1 (mean annual temp)"]
gwa = by_name.loc["Global Wind Atlas (10m wind speed)"]
soilgrids = by_name.loc["SoilGrids topsoil pH 0-5cm"]
ceh = by_name.loc["CEH 50m soil (natural capital pedotopes)"]
jh = by_name.loc["James Hutton 1:250,000 soil map (WRB)"]
nbr_mean = by_name.loc["Neighbour mean height (2023, 75m buffer)"]
nbr_diff = by_name.loc["Neighbour height differential (2023, 75m buffer)"]
dist_cpmt = by_name.loc["Distance to compartment boundary"]

haduk = by_name.loc["HadUK-Grid (tas, 1km)"]

print("1. HadUK-Grid AND CHELSA, same nominal 1km grid, both now real -- neither is 'basically flat':")
print(f"   HadUK-Grid (2021 single-year mean): {int(haduk['n_distinct_values'])} distinct values, "
      f"Moran's I={haduk['morans_i']:.2f}, between-compartment ICC={haduk['between_compartment_icc']:.2f}.")
print(f"   CHELSA (1981-2010, 30-year climatology): {int(chelsa['n_distinct_values'])} distinct values, "
      f"Moran's I={chelsa['morans_i']:.2f}, between-compartment ICC={chelsa['between_compartment_icc']:.2f}.")
print(f"   Both flatly disprove the plan's original '~2-3 cells' assumption for HadUK-Grid (149, not "
      f"2-3) -- but the two aren't apples-to-apples: HadUK-Grid's much higher count is plausibly "
      f"driven largely by one real year's weather variability, not finer spatial downscaling, while "
      f"CHELSA is a smoothed 30-year average by construction. A fair test of 'does terrain-conditioned "
      f"downscaling add real detail' would need several HadUK-Grid years averaged first, not done "
      f"here -- only one 2021 file has been downloaded so far (see notes on this row).\n")

print("2. James Hutton's '~3-5 polygons total' does not survive contact with real data:")
print(f"   {int(jh['n_distinct_values'])} distinct World Reference Base soil groups found in just a "
      f"150-point sample (not the full 71,766) -- more classes than the plan assumed cover the "
      f"ENTIRE study area. Still coarse (1:250,000 scale), but 'coarse polygons' and 'only 3-5 "
      f"values' are different claims, and only the first holds up.\n")

print("3. Global Wind Atlas has by far the most local texture of any external source tested:")
print(f"   {int(gwa['n_distinct_values']):,} distinct wind-speed values, and the LOWEST "
      f"between-compartment ICC ({gwa['between_compartment_icc']:.2f}) of any real-valued external "
      f"covariate -- meaning more of its variation happens WITHIN a compartment, not just between "
      f"compartments, which is exactly the scale the dissertation's attribution question needs.\n")

print("4. The neighbour-height spatial-lag warning is not theoretical -- it shows up directly:")
print(f"   neighbour_mean_height correlates with the CR residual at Spearman r={nbr_mean['spearman_vs_cr_residual_4survey']:.3f} "
      f"(p<0.001) -- by far the strongest of any covariate tested here, external or derived. "
      f"Exactly the risk flagged in the task: this feature could silently carry most of the "
      f"'explanatory power' in a naive terrain+wind+neighbour SHAP run, without it being a real "
      f"environmental driver at all. The with/without SHAP check is not optional for this feature.\n")

print("5. neighbour_height_differential is the most purely WITHIN-compartment covariate found:")
print(f"   between-compartment ICC={nbr_diff['between_compartment_icc']:.3f} (~{nbr_diff['between_compartment_icc']*100:.0f}% "
      f"of its variance is between compartments -- almost none), vs. distance_to_cpmt_boundary's "
      f"{dist_cpmt['between_compartment_icc']:.3f}. Genuinely novel technical finding: the smooth "
      f"climate/soil products above (ICC 0.5-0.93) structurally CANNOT explain within-stand "
      f"variation the way this feature's shape allows -- but it inherits the same spatial-lag "
      f"caveat as point 4, so this is a capability note, not a green light to use it unchecked.")


## What this means for `dnn_env_terrain` / `pinn_env_terrain`

**Corrected below** -- this cell originally said HadUK-Grid/ERA5-Land/AlphaEarth were "still
blocked," written before the mid-session GEE project-ID fix (see "Update: AlphaEarth and
ERA5-Land unblocked mid-session" above) and HadUK-Grid's separate manual-download fix were both
applied later in this same notebook. Left uncorrected, it directly caused a real mix-up outside
this notebook (a summary elsewhere was written from this cell's wording, not the actual
extraction cells). Fixed now to match what the notebook actually shows.

Ready to use now, backed by real extraction and a real statistical screen (not resolution
labels): OS Terrain 50 terrain group (from the prior notebook), Global Wind Atlas, SoilGrids pH,
CHELSA bio1, CEH pedotope class, and the three genuinely-exogenous new GPKG features (distance
to compartment boundary, distance to forest perimeter, elevation roughness).

**Held back, not included in the first terrain+wind feature set:** neighbour mean height and
neighbour height differential -- real signal, but only usable after the with/without SHAP check
this notebook's findings say is now necessary, not optional.

**Access is no longer the reason these three are excluded -- each has its own real, separate
limitation instead:**
- **HadUK-Grid** -- real extraction confirmed (149 distinct values, Moran's I=0.94, and the
  *strongest* Spearman-vs-residual correlation of anything tested, 0.291). Only one year (2021)
  has actually been downloaded so far, though -- that's one year's weather variability, not
  necessarily real spatial detail. Needs several years averaged before it's trustworthy enough
  to promote to the main list, not blocked by access anymore.
- **ERA5-Land** -- real GEE extraction confirmed (~11.1km native resolution, only 8 distinct
  values across the whole forest, ICC=0.903). Access works now, but it's genuinely coarse and
  weak (Spearman 0.106) next to CHELSA/HadUK-Grid, which already cover temperature at finer
  resolution -- left off the main list on information-content grounds, not an access block.
- **AlphaEarth** -- access fixed via the same GEE project ID. Still excluded, but for a
  different, unfixable-by-access reason: 2017-onward coverage only, which can't support this
  study's 2002-2023 span regardless of whether the API works.

**Still genuinely blocked:** WASP only (contact request to Dr Suárez-Minguez, per the prior
notebook, still not made) -- a people-dependency, not a technical one.

***

# Part 4: Do the wind/exposure variables actually agree with each other?

TOPEX and Global Wind Atlas wind speed both claim to measure "how windy/exposed is this plot",
but they're built completely differently -- TOPEX is a terrain-shape calculation from the DTM,
Global Wind Atlas is an independent mesoscale wind model. This section checks whether they
actually agree, and builds one new feature: a **prevailing-wind-direction** version of TOPEX,
instead of the omnidirectional 8-direction version used above.

**Important sign-convention reminder, since it's easy to misread**: in this notebook,
**positive TOPEX = sheltered** (surrounded by higher ground), **negative TOPEX = exposed**
(see the synthetic-hill validation earlier in this notebook, where the summit scored a strongly
negative -133). Every comparison below uses that same convention -- keep it in mind when reading
the correlation signs.

In [ ]:
# The notebook so far has TPI and roughness (both DERIVED from elevation), but never saved the
# plain elevation value itself at each plot. Add it now, using the exact same lookup pattern
# (mosaic + row_idx/col_idx) already used for every other DTM-derived feature above.
plots["elevation"] = np.nan
plots.loc[in_bounds, "elevation"] = mosaic[row_idx[in_bounds], col_idx[in_bounds]]

print(f"Elevation: {plots['elevation'].min():.1f}m to {plots['elevation'].max():.1f}m, "
      f"mean {plots['elevation'].mean():.1f}m")


## 4.1 How much do the exposure-family variables actually agree with each other?

Not each one against the CR residual (already done above) -- this time, each one against the
OTHERS, to see which pairs are measuring similar things and which are picking up something
different. A simple Spearman correlation matrix, same correlation method used everywhere else
in this notebook for consistency.

In [ ]:
# List every "how exposed/windy is this plot" variable built so far, plus elevation itself
# (since exposure and elevation are often related in real terrain).
exposure_family_columns = ["topex", "gwa_wind_speed_10m", "elevation_roughness", "tpi", "elevation"]

# Build a small table with just these columns, dropping any row where one of them is missing --
# a correlation needs every column to have a real number in the same rows.
exposure_family_table = plots[exposure_family_columns].dropna()
print(f"Rows with all five exposure-family values present: {len(exposure_family_table):,}")

# .corr(method="spearman") computes every pairwise correlation at once, which is exactly the
# same statistic (spearmanr) used one pair at a time everywhere else in this notebook.
exposure_family_correlations = exposure_family_table.corr(method="spearman")
display(exposure_family_correlations.style.format("{:.3f}").background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))


## 4.2 Controlling for elevation -- is the TOPEX/GWA-vs-residual relationship still there?

TOPEX and elevation are related (higher land tends to be more exposed), and elevation itself
might correlate with the CR residual for reasons that have nothing to do with wind (for example,
if higher plots happen to have been surveyed in different years, or are systematically
younger/older). This checks whether TOPEX's and GWA's relationship with the residual survives
once elevation's own effect is removed first -- a **partial correlation**.

**How a partial correlation works, in plain terms**: fit a straight line predicting TOPEX from
elevation, and look at what's LEFT OVER (the leftover part is called the "residual" of that
line -- a different use of the word "residual" than the CR growth-curve residual, worth not
confusing the two). Do the same for the CR residual predicted from elevation. Then correlate the
two leftover parts against each other. Whatever correlation is left after this is the part of
the TOPEX-vs-CR-residual relationship that elevation alone can't explain.

In [ ]:
def remove_elevation_effect(values, elevation):
    # Fits a simple straight-line (linear) relationship: values = slope * elevation + intercept.
    # Returns the "leftover" part of `values` that this straight line does NOT explain --
    # i.e. actual value minus what the line predicts. This is the standard, simplest way to
    # "control for" a variable before checking a different correlation.
    valid = (~np.isnan(values)) & (~np.isnan(elevation))
    slope, intercept = np.polyfit(elevation[valid], values[valid], deg=1)
    predicted = slope * elevation[valid] + intercept
    leftover = values[valid] - predicted
    return leftover, valid


def partial_correlation_controlling_for_elevation(value_col, residual_col, plots_df):
    values = plots_df[value_col].values
    residuals = plots_df[residual_col].values
    elevation = plots_df["elevation"].values

    # Remove elevation's effect from BOTH variables, one at a time.
    value_leftover, value_valid = remove_elevation_effect(values, elevation)
    residual_leftover, residual_valid = remove_elevation_effect(residuals, elevation)

    # Only keep rows where both leftover values exist (both had real elevation + real value).
    both_valid_index = plots_df.index[value_valid].intersection(plots_df.index[residual_valid])
    value_leftover_series = pd.Series(value_leftover, index=plots_df.index[value_valid])
    residual_leftover_series = pd.Series(residual_leftover, index=plots_df.index[residual_valid])

    rho, p_value = spearmanr(
        value_leftover_series.loc[both_valid_index], residual_leftover_series.loc[both_valid_index],
    )
    return rho, p_value, len(both_valid_index)


print("Partial correlation with CR residual (4survey), controlling for elevation:")
for covariate_name in ["topex", "gwa_wind_speed_10m"]:
    rho, p_value, n_rows = partial_correlation_controlling_for_elevation(
        covariate_name, "mean_cr_residual_4survey", plots,
    )
    # Compare against the raw (non-elevation-controlled) correlation already computed earlier
    # in this notebook, so it's easy to see how much elevation was doing.
    raw_rho, raw_p = spearman_vs_residual(plots[covariate_name].values, plots["mean_cr_residual_4survey"].values)
    print(f"  {covariate_name}: raw rho={raw_rho:.3f} (p={raw_p:.3f})  ->  "
          f"elevation-controlled rho={rho:.3f} (p={p_value:.3f}), n={n_rows:,}")


## 4.3 A prevailing-wind-direction version of TOPEX, not omnidirectional

The TOPEX built earlier in this notebook sums horizon angles from **all 8 compass directions
equally**. But real windthrow/exposure literature is clear that windward and leeward slopes
behave differently -- a slope facing away from the prevailing wind is genuinely more sheltered
than one facing into it, and an omnidirectional average can't tell the two apart.

Scotland's prevailing wind is from the **south-west** -- conveniently, 225 degrees is already
one of the 8 directions the existing `topex()` function samples, so this needs no new geometry
code at all: just call the same, already-validated function again, with only one direction
instead of eight.

In [ ]:
# Re-use the exact same topex() function from earlier in this notebook -- just pass a single
# direction (225 degrees = south-west, Scotland's prevailing wind) instead of all eight.
PREVAILING_WIND_BEARING_DEG = 225

windward_topex_values = topex(mosaic, directions_deg=[PREVAILING_WIND_BEARING_DEG])
plots["windward_topex"] = np.nan
plots.loc[in_bounds, "windward_topex"] = windward_topex_values[row_idx[in_bounds], col_idx[in_bounds]]

print(f"Windward-only TOPEX (225 degrees / SW): "
      f"{plots['windward_topex'].min():.1f} (most exposed) to {plots['windward_topex'].max():.1f} (most sheltered)")

row = screen_covariate(
    plots, "windward_topex", name="Windward TOPEX (225 deg SW only, 1000m radius)",
    category="terrain-derived (new, from OS Terrain 50)",
    access_method="Same DTM mosaic and topex() function already in this notebook, single direction",
    years_covered="All 6 -- DTM is static",
    notes="Prevailing-wind-direction version of the omnidirectional TOPEX above -- tests "
          "whether windward/leeward matters more than average exposure for this forest.",
)
results.append(row)


In [ ]:
# Does the windward-only version agree with Global Wind Atlas any better than the
# omnidirectional TOPEX did? Same Spearman correlation method as the rest of this notebook.
omni_vs_gwa_rho, omni_vs_gwa_p = spearmanr(plots["topex"], plots["gwa_wind_speed_10m"], nan_policy="omit")
windward_vs_gwa_rho, windward_vs_gwa_p = spearmanr(plots["windward_topex"], plots["gwa_wind_speed_10m"], nan_policy="omit")

windward_vs_residual_rho, windward_vs_residual_p = spearman_vs_residual(
    plots["windward_topex"].values, plots["mean_cr_residual_4survey"].values,
)

print(f"Omnidirectional TOPEX vs Global Wind Atlas:  rho={omni_vs_gwa_rho:.3f} (p={omni_vs_gwa_p:.3f})")
print(f"Windward-only TOPEX vs Global Wind Atlas:    rho={windward_vs_gwa_rho:.3f} (p={windward_vs_gwa_p:.3f})")
print(f"Windward-only TOPEX vs CR residual (4survey): rho={windward_vs_residual_rho:.3f} (p={windward_vs_residual_p:.3f})")


**Reading this section**: if the windward-only version correlates with Global Wind Atlas more
strongly than the omnidirectional version does, that's real evidence the leeward/windward
distinction matters here, not just in the general literature -- worth carrying the
windward-only version forward into SHAP/NLME instead of (or alongside) the omnidirectional one.
If the two TOPEX versions barely differ, that's useful too -- it means Aberfoyle's terrain
doesn't have enough of a directional asymmetry for this specific refinement to matter, and the
simpler omnidirectional version is good enough.